In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports and Environment ===
# Imports, environment check, plot style and image helpers. No parameters.

import os
import sys
import json
import time
import shutil
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Polygon as MplPolygon
from PIL import Image, PngImagePlugin

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from scipy import ndimage

# --- Optional dependencies ---
try:
    import cv2
except ImportError:
    cv2 = None

try:
    import torch
except ImportError:
    torch = None

# pycocotools is required: PackWISE masks are compressed COCO RLE (byte-encoded counts),
# for which there is no reliable pure-numpy decoder.
try:
    from pycocotools import mask as maskUtils
except ImportError:
    maskUtils = None

Image.MAX_IMAGE_PIXELS = None                 # 4096x4096 line-scan frames
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

# --- Plot style ---
_FW = 16 / 2.54          # LaTeX \textwidth in inches
_FONT_SIZE = 10

_PREFERRED_FONTS = ["DejaVu Sans", "Helvetica", "Arial", "Nimbus Sans",
                    "TeX Gyre Heros", "Liberation Sans"]
_available = {f.name for f in fm.fontManager.ttflist}
_font = next((f for f in _PREFERRED_FONTS if f in _available), "DejaVu Sans")

plt.rcParams["font.family"] = _font
plt.rcParams["font.size"] = _FONT_SIZE
plt.rcParams["axes.titlesize"] = _FONT_SIZE + 1
plt.rcParams["axes.labelsize"] = _FONT_SIZE
plt.rcParams["legend.fontsize"] = _FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = _FONT_SIZE - 1
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["legend.frameon"] = False


def _panel_label(ax, letter, title):
    """Bold panel letter in front of the subtitle."""
    ax.set_title(f"$\\bf{{({letter})}}$ {title}")


def _titled_legend(fig, title, handles=None, ncol=3, top_rect=0.86):
    """Suptitle plus optional figure legend, stacked so they do not overlap."""
    if handles:
        fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.995),
                   ncol=ncol, frameon=False)
        fig.suptitle(title, y=1.09)
    else:
        fig.suptitle(title, y=1.02)
    plt.tight_layout(rect=[0, 0, 1, top_rect])


def load_rgb(path):
    """RGB uint8 array from any PIL-readable file."""
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


def image_size(path):
    """(width, height) without decoding pixels."""
    with Image.open(path) as im:
        return im.size


# --- Environment report ---
print("=" * 78)
print("ENVIRONMENT")
print("=" * 78)
print(f"python        : {sys.executable}")
print(f"python ver    : {sys.version.split()[0]}")
print(f"cwd           : {Path.cwd()}")
print(f"plot font     : {_font}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"opencv        : {getattr(cv2, '__version__', 'NOT INSTALLED  ->  polygon export will fail')}")

if maskUtils is None:
    print("pycocotools   : NOT INSTALLED  ->  Stage 1b CANNOT decode PackWISE RLE masks")
    print("                install with:  pip install pycocotools")
else:
    print("pycocotools   : available")

if torch is None:
    print("torch         : NOT INSTALLED  ->  SAM3 stages will not run")
else:
    print(f"torch         : {torch.__version__}")
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print(f"cuda          : available, {n} device(s)")
        for i in range(n):
            props = torch.cuda.get_device_properties(i)
            print(f"   [{i}] {props.name}  {props.total_memory / 1024**3:.0f} GB")
    else:
        print("cuda          : NOT AVAILABLE  ->  SAM3 on CPU is infeasible at this scale")

# Subprocess workers run with sys.executable, so the kernel must be the sam3-seg env.
if "sam3-seg" not in sys.executable:
    print()
    print("!! WARNING: interpreter does not look like the 'sam3-seg' env.")
    print("!! Expected something like /home/<user>/.conda/envs/sam3-seg/bin/python")
print("=" * 78)

In [ ]:
# === Cell 3: Config ===
# Paths, compute settings and fixed invariants. Tunable parameters are set via widgets in
# the stage cells and frozen into STAGE_PARAMS by each stage's Lock button.

# --- Run identity ---
RUN_TAG = "packwise_v1"
RUN_ID = f"{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"

# --- Input ---
DATASET_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/PackWISE_Dataset_v2")
PACKWISE_DATA = DATASET_ROOT / "data"          # holds train/ val/ test/ and *.json
SPLIT_FILES = {"train": PACKWISE_DATA / "train.json",
               "val": PACKWISE_DATA / "val.json",
               "test": PACKWISE_DATA / "test.json"}

# --- Output ---
OUT_ROOT = DATASET_ROOT / "KI_Sort_Preprocessing"
DATA_OUT_ROOT = OUT_ROOT / "data" / RUN_ID
RESULTS_OUT_ROOT = OUT_ROOT / "results" / RUN_ID

DIR_TABLES = RESULTS_OUT_ROOT / "tables"
DIR_LOGS = RESULTS_OUT_ROOT / "logs"
DIR_WORKERS = RESULTS_OUT_ROOT / "workers"
DIR_FIGURES = RESULTS_OUT_ROOT / "figures"
DIR_STAGE1 = DATA_OUT_ROOT / "stage1b_whole_masks"
DIR_CUTOUTS = DATA_OUT_ROOT / "stage2_cutouts"
DIR_PSEUDOGT = DATA_OUT_ROOT / "stage3_pseudo_gt"
DIR_YOLO_DS = DATA_OUT_ROOT / "yolo_dataset"

for _d in (DIR_TABLES, DIR_LOGS, DIR_WORKERS, DIR_FIGURES, DIR_STAGE1, DIR_CUTOUTS,
           DIR_PSEUDOGT, DIR_YOLO_DS):
    _d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = RESULTS_OUT_ROOT / "run_manifest.json"

# --- Models ---
SAM3_WEIGHTS = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/08_Generalizability/sam3_weights/sam3.pt")

# Deployment model, used only for visual QA in the YOLO Visual QA cell. It is trained on
# all data, so no metrics are computed against it here; scoring is done in the benchmark
# notebook.
YOLO_DEPLOY_WEIGHTS = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260603_Yolo_Deployment_Training/"
    "runs/20260603_Yolo_Deployment_Training_DeploymentSegTraining/weights/best.pt")

# --- KI-Sort invariants ---
CLASS_MAP = {"body": 0, "cap": 1, "label": 2}
CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}
REGION_ORDER = ("body", "label", "cap")
REGION_COLORS = {"body": "#FF0000", "cap": "#00FF00", "label": "#0000FF"}
C_BODY, C_LABEL, C_CAP = REGION_COLORS["body"], REGION_COLORS["label"], REGION_COLORS["cap"]

# Deployment model runs at imgsz=640, rect=True -> inference shape 384x640 (h x w).
# Cutout canvases use this aspect and size, so no letterbox padding is introduced.
YOLO_INFER_H, YOLO_INFER_W = 384, 640
CANVAS_ASPECT_W, CANVAS_ASPECT_H = YOLO_INFER_W, YOLO_INFER_H

# --- In-scope criterion ---
# Explicit in/out decision with a stated reason for each of the 23 PackWISE categories.
# Written verbatim to the manifest and the dataset card.
CATEGORY_DECISIONS = {
    # In scope: rigid plastic packaging with a body and an optional closure
    "plastic-bottle":        (True,  "rigid plastic container, body + optional closure"),
    "plastic-cup/pot":       (True,  "rigid plastic container, body + optional closure"),
    "blister":               (True,  "rigid plastic pack with a body and printed backing"),
    "tube":                  (True,  "rigid plastic container, body + closure"),
    # Out of scope: plastic, but not a whole container
    "plastic-lid":           (False, "detached closure annotated as its own instance; has "
                                     "no body/label decomposition, so the KI-Sort region "
                                     "model does not apply"),
    "plastic-foil/bag":      (False, "flexible film, not a rigid container"),
    "foamed-plastic":        (False, "foam fragment, not a packaging container form"),
    "plastic-other":         (False, "non-specific catch-all; membership cannot be stated, "
                                     "so it cannot be defended as in scope"),
    "undefined":             (False, "unlabelled residual category"),
    # Out of scope: non-polymer or composite material streams
    "aluminium-coated-foil": (False, "metallised composite"),
    "paper-bag":             (False, "fibre-based"),
    "beverage-carton":       (False, "fibre composite"),
    "metal-can":             (False, "metal"),
    "metal-foil":            (False, "metal"),
    "metal-lid":             (False, "metal"),
    "metal-other":           (False, "metal"),
    "net":                   (False, "mesh, not a rigid container"),
    "tissue":                (False, "fibre-based"),
    "wood":                  (False, "wood"),
    "paper-other":           (False, "fibre-based"),
    "paper-cup/pot":         (False, "fibre-based"),
    "textile":               (False, "textile"),
    "ribbon/tape":           (False, "not packaging body form"),
}

IN_SCOPE_CRITERION = (
    "Rigid plastic packaging presenting a body with an optional closure: PackWISE "
    "categories plastic-bottle, plastic-cup/pot, blister, tube. All other categories are "
    "excluded with a per-category stated reason (see CATEGORY_DECISIONS). Note in "
    "particular that plastic-lid is excluded despite being plastic: it is annotated as a "
    "detached closure with no body or label, so the KI-Sort body/cap/label decomposition "
    "has no meaning for it. KI-Sort's polymer scope is HDPE/PET/PP/PS."
)


def is_in_scope(class_name):
    return bool(CATEGORY_DECISIONS.get(str(class_name).strip(), (False, ""))[0])


# --- Compute ---
GPU_POLICY = "auto"        # auto | single | multi | cpu
GPU_IDS = None             # None -> all visible devices
PROGRESS_EVERY = 25
RATE_WINDOW = 40
WORKER_POLL_S = 5.0
RESUME = True
EMBED_PROVENANCE = True
CUTOUT_FORMAT = "png"      # lossless: cutout pixels are re-read by Stage 3

# --- Config record ---
CONFIG = {
    "run_tag": RUN_TAG,
    "dataset_root": str(DATASET_ROOT),
    "packwise_data": str(PACKWISE_DATA),
    "data_out_root": str(DATA_OUT_ROOT),
    "results_out_root": str(RESULTS_OUT_ROOT),
    "sam3_weights": str(SAM3_WEIGHTS),
    "yolo_deploy_weights": str(YOLO_DEPLOY_WEIGHTS),
    "class_map": CLASS_MAP,
    "yolo_infer_hw": [YOLO_INFER_H, YOLO_INFER_W],
    "canvas_aspect": [CANVAS_ASPECT_W, CANVAS_ASPECT_H],
    "in_scope_criterion": IN_SCOPE_CRITERION,
    "category_decisions": {k: {"in_scope": v[0], "reason": v[1]}
                           for k, v in CATEGORY_DECISIONS.items()},
    "cutout_format": CUTOUT_FORMAT,
    "gpu_policy": GPU_POLICY,
}
CONFIG_HASH = hashlib.sha1(
    json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:12]

print("=" * 78)
print("CONFIGURATION")
print("=" * 78)
print(f"run id        : {RUN_ID}")
print(f"config hash   : {CONFIG_HASH}")
print(f"dataset root  : {DATASET_ROOT}")
for name, path in SPLIT_FILES.items():
    print(f"  {name:<6}      : {'OK' if path.exists() else 'MISSING'}  {path.name}")
print(f"data out      : {DATA_OUT_ROOT}")
print(f"results out   : {RESULTS_OUT_ROOT}")
print(f"sam3 weights  : {'OK' if SAM3_WEIGHTS.exists() else 'MISSING'}")
print(f"yolo weights  : {'OK' if YOLO_DEPLOY_WEIGHTS.exists() else 'MISSING'}")
print(f"canvas aspect : {CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}   export {YOLO_INFER_W}x{YOLO_INFER_H}")
_n_in = sum(1 for v in CATEGORY_DECISIONS.values() if v[0])
print(f"categories    : {_n_in} in scope / {len(CATEGORY_DECISIONS)} decided")
print("=" * 78)

In [ ]:
# === Cell 4: Pipeline Infrastructure ===
# Manifest, parameter locking, progress/ETA, compute plan, checkpoints, provenance and
# widget helpers. Dataset-independent; shared with the other preprocessing notebooks.

# --- Small utilities ---

def slugify(text):
    """Filesystem-safe lowercase token (non-alphanumerics become underscores)."""
    keep = [c.lower() if c.isalnum() else "_" for c in str(text)]
    out = "".join(keep)
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def fmt_dur(seconds):
    if seconds is None or not np.isfinite(seconds):
        return "--"
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"


def fmt_bytes(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"


def hash_params(params):
    return hashlib.sha1(json.dumps(params, sort_keys=True, default=str).encode()).hexdigest()[:12]


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


# --- Manifest ---

def read_manifest():
    """Read the run manifest; an unreadable file is backed up and treated as empty."""
    if not MANIFEST_PATH.exists():
        return {}
    try:
        return json.loads(MANIFEST_PATH.read_text())
    except (json.JSONDecodeError, ValueError):
        broken = MANIFEST_PATH.with_suffix(".json.broken")
        shutil.copy2(MANIFEST_PATH, broken)
        print(f"!! manifest unreadable, copied to {broken.name} and starting fresh")
        return {}


def write_manifest(man):
    """Atomic: write to a temp file in the same dir, then replace."""
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(man, indent=2, sort_keys=True, default=str))
    tmp.replace(MANIFEST_PATH)


def log_params(stage, **kv):
    """Merge a stage record into the run manifest."""
    man = read_manifest()
    man.setdefault("run_id", RUN_ID)
    man.setdefault("run_tag", RUN_TAG)
    man.setdefault("config_hash", CONFIG_HASH)
    man.setdefault("config", CONFIG)
    man.setdefault("created_utc", utcnow())
    man["updated_utc"] = utcnow()
    stages = man.setdefault("stages", {})
    rec = stages.setdefault(stage, {})
    for k, v in kv.items():
        if k == "params" and isinstance(v, dict):
            rec.setdefault("params", {}).update(v)
        else:
            rec[k] = v
    write_manifest(man)
    return man


# --- Parameter locking ---
# Widget values are frozen and logged per stage so batch runs are reproducible.

STAGE_PARAMS = {}   # stage -> {"params": {...}, "param_hash": ..., "locked_utc": ...}


def lock_stage(stage, params, n_preview=None, notes=None):
    """Freeze widget values, log them, and authorise the stage's batch run."""
    entry = {
        "params": dict(params),
        "param_hash": hash_params(params),
        "locked_utc": utcnow(),
        "n_preview": n_preview,
    }
    if notes:
        entry["notes"] = notes
    STAGE_PARAMS[stage] = entry
    log_params(stage, **entry)
    print(f"[{stage}] LOCKED  hash={entry['param_hash']}")
    for k, v in sorted(params.items()):
        print(f"    {k:24s} = {v}")
    return entry


def require_locked(stage):
    """Return the locked params for a stage; raise if the stage is not locked."""
    if stage not in STAGE_PARAMS:
        raise RuntimeError(
            f"'{stage}' has no locked parameters.\n"
            f"Run its preview cell and press 'Lock params' before starting the batch."
        )
    return dict(STAGE_PARAMS[stage]["params"])


def load_params_from_run(run_id, stages=None, results_root=None):
    """Repopulate STAGE_PARAMS from a previous run's manifest."""
    root = Path(results_root) if results_root else RESULTS_OUT_ROOT.parent
    path = root / run_id / "run_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"No manifest at {path}")
    man = json.loads(path.read_text())
    loaded = {}
    for stage, rec in man.get("stages", {}).items():
        if stages and stage not in stages:
            continue
        if "params" in rec:
            STAGE_PARAMS[stage] = {
                "params": rec["params"],
                "param_hash": rec.get("param_hash"),
                "locked_utc": rec.get("locked_utc"),
                "n_preview": rec.get("n_preview"),
                "loaded_from": run_id,
            }
            loaded[stage] = rec["params"]
    print(f"Loaded {len(loaded)} stage param set(s) from {run_id}: {sorted(loaded)}")
    return loaded


# --- Progress and ETA ---

class ProgressReporter:
    """Rolling-window throughput and ETA.

    A rolling window is used because the first items include model load and GPU warm-up.
    """

    def __init__(self, stage, n_total, every=None, window=None, log_path=None, quiet=False):
        self.stage = stage
        self.n_total = max(int(n_total), 0)
        self.every = every or PROGRESS_EVERY
        self.window = window or RATE_WINDOW
        self.quiet = quiet
        self.n_done = 0
        self.t0 = time.time()
        self._samples = deque(maxlen=self.window)     # (timestamp, n_done)
        self._samples.append((self.t0, 0))
        self._throughput = []
        self.log_path = Path(log_path) if log_path else DIR_LOGS / f"{stage}_progress.log"
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self._fh = open(self.log_path, "a", buffering=1)
        self._emit(f"--- {stage} start {utcnow()}  n_total={self.n_total} ---")

    def _rate(self):
        if len(self._samples) < 2:
            return 0.0
        t_old, n_old = self._samples[0]
        t_new, n_new = self._samples[-1]
        dt = t_new - t_old
        return (n_new - n_old) / dt if dt > 0 else 0.0

    def _emit(self, line):
        self._fh.write(line + "\n")
        if not self.quiet:
            print(line)

    def update(self, n=1, force=False):
        self.n_done += n
        self._samples.append((time.time(), self.n_done))
        if force or self.n_done % self.every == 0 or self.n_done == self.n_total:
            self.report()

    def set_done(self, n_done, force=False):
        """Set an absolute done count (used when aggregating subprocess workers)."""
        self.n_done = int(n_done)
        self._samples.append((time.time(), self.n_done))
        if force or self.n_done >= self.n_total:
            self.report()
        return self.n_done

    def report(self):
        elapsed = time.time() - self.t0
        rate = self._rate()
        remaining = max(self.n_total - self.n_done, 0)
        eta = remaining / rate if rate > 1e-9 else float("inf")
        pct = 100.0 * self.n_done / self.n_total if self.n_total else 0.0
        line = (f"[{self.stage}] {self.n_done}/{self.n_total} ({pct:5.1f}%) | "
                f"{rate:6.2f} it/s | elapsed {fmt_dur(elapsed)} | ETA {fmt_dur(eta)}")
        self._emit(line)
        self._throughput.append({"t": round(elapsed, 1), "n": self.n_done, "rate": round(rate, 4)})

    def close(self, extra=None):
        elapsed = time.time() - self.t0
        self.report()
        self._emit(f"--- {self.stage} done {utcnow()}  {self.n_done}/{self.n_total} "
                   f"in {fmt_dur(elapsed)} ---")
        self._fh.close()
        rec = {
            "n_total": self.n_total,
            "n_done": self.n_done,
            "elapsed_s": round(elapsed, 1),
            "mean_rate_it_s": round(self.n_done / elapsed, 4) if elapsed > 0 else None,
            "throughput_samples": self._throughput[-200:],
            "progress_log": str(self.log_path),
        }
        if extra:
            rec.update(extra)
        log_params(self.stage, runtime=rec)
        return rec


def estimate_batch_runtime(seconds_per_item, n_items):
    """Projected batch runtime from preview throughput."""
    if not seconds_per_item or not np.isfinite(seconds_per_item):
        return "unknown"
    return fmt_dur(seconds_per_item * n_items)


# --- Compute plan ---

def resolve_compute_plan(policy=None, gpu_ids=None):
    policy = policy or GPU_POLICY
    ids = gpu_ids if gpu_ids is not None else GPU_IDS

    n_cuda = 0
    names = []
    if torch is not None and torch.cuda.is_available():
        n_cuda = torch.cuda.device_count()
        names = [torch.cuda.get_device_properties(i).name for i in range(n_cuda)]

    if ids is None:
        ids = list(range(n_cuda))
    else:
        ids = [i for i in ids if i < n_cuda]

    if policy == "cpu" or n_cuda == 0:
        mode, devices = "cpu", []
    elif policy == "single":
        mode, devices = "single", ids[:1]
    elif policy == "multi":
        mode, devices = ("multi", ids) if len(ids) > 1 else ("single", ids[:1])
    else:  # auto
        mode, devices = ("multi", ids) if len(ids) > 1 else ("single", ids[:1])

    return {
        "mode": mode,
        "devices": devices,
        "n_workers": max(len(devices), 1),
        "device_names": [names[i] for i in devices] if devices else [],
        "policy": policy,
        "n_cuda_visible": n_cuda,
    }


COMPUTE_PLAN = resolve_compute_plan()

print("=" * 78)
print("COMPUTE PLAN")
print("=" * 78)
print(f"policy        : {COMPUTE_PLAN['policy']}")
print(f"mode          : {COMPUTE_PLAN['mode']}")
print(f"cuda devices  : {COMPUTE_PLAN['devices'] or 'none'}")
for i, nm in zip(COMPUTE_PLAN["devices"], COMPUTE_PLAN["device_names"]):
    print(f"   [{i}] {nm}")
print(f"workers       : {COMPUTE_PLAN['n_workers']}")
if COMPUTE_PLAN["mode"] == "cpu":
    print("!! CPU mode: SAM3 encode is ~260 s/image. A full-archive run is not feasible.")
print("=" * 78)


# --- Checkpoints ---

def checkpoint_path(stage):
    return DIR_TABLES / f"{stage}_checkpoint.csv"


def load_checkpoint(stage, key_col="item_id"):
    """Returns (DataFrame, set_of_completed_keys). Empty if absent or RESUME is off."""
    p = checkpoint_path(stage)
    if not RESUME or not p.exists():
        return pd.DataFrame(), set()
    try:
        df = pd.read_csv(p)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        return pd.DataFrame(), set()
    done = set(df[key_col].astype(str)) if key_col in df.columns else set()
    return df, done


def append_checkpoint(stage, rows):
    """Append rows to the stage checkpoint. Header written only on creation."""
    if not rows:
        return
    p = checkpoint_path(stage)
    df = pd.DataFrame(rows)
    df.to_csv(p, mode="a", header=not p.exists(), index=False)


def count_checkpoint_rows(path):
    """Number of data rows in a checkpoint CSV; used to poll subprocess workers."""
    p = Path(path)
    if not p.exists():
        return 0
    try:
        with open(p, "rb") as fh:
            n = sum(1 for _ in fh)
        return max(n - 1, 0)       # minus header
    except OSError:
        return 0


def count_checkpoint_items(path, key_col="item_id"):
    """Number of distinct `key_col` values in a checkpoint CSV.

    Stages that write several rows per item need this instead of a raw row count."""
    p = Path(path)
    if not p.exists():
        return 0
    try:
        df = pd.read_csv(p, usecols=lambda c: c == key_col)
        if key_col not in df.columns:
            return count_checkpoint_rows(path)
        return int(df[key_col].astype(str).nunique())
    except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError, ValueError):
        return 0


# --- Provenance ---
# Provenance is embedded in mask/image files and also written to per-stage CSVs.

def provenance_record(stage, source_paths, params, extra=None):
    rec = {
        "run_id": RUN_ID,
        "stage": stage,
        "config_hash": CONFIG_HASH,
        "param_hash": hash_params(params),
        "created_utc": utcnow(),
    }
    if isinstance(source_paths, dict):
        rec.update({f"source_{k}": str(v) for k, v in source_paths.items()})
    else:
        rec["source_path"] = str(source_paths)
    if extra:
        rec.update(extra)
    return rec


def save_mask_npz(path, provenance=None, **arrays):
    """Save masks as compressed .npz with an embedded provenance record."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(arrays)
    if EMBED_PROVENANCE and provenance is not None:
        payload["provenance"] = np.array(json.dumps(provenance, default=str))
    np.savez_compressed(path, **payload)
    return path


def read_mask_npz(path):
    """Returns (dict_of_arrays, provenance_dict_or_None)."""
    with np.load(path, allow_pickle=False) as z:
        arrays = {k: z[k] for k in z.files if k != "provenance"}
        prov = None
        if "provenance" in z.files:
            try:
                prov = json.loads(str(z["provenance"]))
            except (json.JSONDecodeError, ValueError):
                prov = None
    return arrays, prov


def save_image_with_provenance(path, arr, provenance=None, fmt=None):
    """Save an image; PNG stores provenance in tEXt chunks, JPEG in a sidecar .json."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fmt = (fmt or path.suffix.lstrip(".") or CUTOUT_FORMAT).lower()
    img = Image.fromarray(arr)
    if fmt == "png":
        info = PngImagePlugin.PngInfo()
        if EMBED_PROVENANCE and provenance:
            for k, v in provenance.items():
                info.add_text(str(k), str(v))
        img.save(path, format="PNG", pnginfo=info, optimize=False)
    else:
        img.save(path, format="JPEG", quality=95)
        if EMBED_PROVENANCE and provenance:
            path.with_suffix(".json").write_text(json.dumps(provenance, indent=2, default=str))
    return path


def read_image_provenance(path):
    path = Path(path)
    if path.suffix.lower() == ".png":
        with Image.open(path) as im:
            return dict(im.text) if hasattr(im, "text") else {}
    side = path.with_suffix(".json")
    return json.loads(side.read_text()) if side.exists() else {}


def write_provenance_csv(stage, rows):
    if not rows:
        return None
    p = DIR_TABLES / f"{stage}_provenance.csv"
    pd.DataFrame(rows).to_csv(p, mode="a", header=not p.exists(), index=False)
    return p


# --- Widget helpers ---

def status_html(stage, n_scope=None, est=None, extra=""):
    locked = STAGE_PARAMS.get(stage)
    lock_txt = (f"<span style='color:#2a9d8f'>locked {locked['param_hash']}</span>"
                if locked else "<span style='color:#d1495b'>not locked</span>")
    parts = [f"<b>{stage}</b>", lock_txt]
    if n_scope is not None:
        parts.append(f"scope: {n_scope:,}")
    if est:
        parts.append(f"est. batch: {est}")
    if extra:
        parts.append(extra)
    return ("<div style='font-family:monospace;font-size:12px;padding:4px 8px;"
            "background:#f8fafc;border:1px solid #d1d5db;border-radius:6px'>"
            + " &nbsp;|&nbsp; ".join(parts) + "</div>")


def make_action_bar(on_preview, on_lock, on_batch, batch_label="Run batch"):
    """Uniform [Preview] [Lock params] [Run batch] bar with the enable-chain wired.

    Lock is enabled only after a preview, batch only after locking.
    """
    b_prev = widgets.Button(description="Preview", button_style="info", icon="eye")
    b_lock = widgets.Button(description="Lock params", button_style="warning",
                            icon="lock", disabled=True)
    b_run = widgets.Button(description=batch_label, button_style="success",
                           icon="play", disabled=True)

    def _prev(_):
        on_preview()
        b_lock.disabled = False

    def _lock(_):
        on_lock()
        b_run.disabled = False

    def _run(_):
        b_run.disabled = True
        try:
            on_batch()
        finally:
            b_run.disabled = False

    b_prev.on_click(_prev)
    b_lock.on_click(_lock)
    b_run.on_click(_run)
    return widgets.HBox([b_prev, b_lock, b_run]), (b_prev, b_lock, b_run)


def invalidate_lock(stage, buttons):
    """Any control change re-arms the chain: lock and batch must be redone."""
    STAGE_PARAMS.pop(stage, None)
    _, b_lock, b_run = buttons
    b_lock.disabled = True
    b_run.disabled = True


log_params("_session", started_utc=utcnow(), compute_plan=COMPUTE_PLAN,
           python=sys.executable)
print(f"\nInfrastructure ready. Manifest -> {MANIFEST_PATH}")

In [ ]:
# === Cell 5: Dataset Index ===
# Builds the instance index from the per-split COCO JSONs (compressed RLE masks per
# instance). The index stores `split` + `ann_id` only; masks are decoded from the JSON.
# Images are 4096x4096, so later pixel stages group their work by image.

INDEX = {"objects": None, "categories": None}


def _load_split(split):
    p = SPLIT_FILES.get(split)
    if p is None or not p.exists():
        return None
    return json.loads(p.read_text())


def _resolve_image_path(split, file_name):
    """Locate an image for a COCO file_name (bare or with directory prefix)."""
    cands = [PACKWISE_DATA / split / Path(file_name).name,
             PACKWISE_DATA / file_name,
             PACKWISE_DATA / split / file_name]
    return next((c for c in cands if c.exists()), None)


def build_object_index():
    rows, cats_seen = [], {}
    for split in SPLIT_FILES:
        d = _load_split(split)
        if d is None:
            continue
        cats = {c["id"]: c["name"] for c in d.get("categories", [])}
        cats_seen.update(cats)
        imgs = {im["id"]: im for im in d.get("images", [])}
        for ann in d.get("annotations", []):
            im = imgs.get(ann.get("image_id"))
            if im is None:
                continue
            img_path = _resolve_image_path(split, im.get("file_name", ""))
            if img_path is None:
                continue
            name = cats.get(ann.get("category_id"), str(ann.get("category_id")))
            seg = ann.get("segmentation")
            bbox = ann.get("bbox", [np.nan] * 4)
            x, y, bw, bh = (list(bbox) + [np.nan] * 4)[:4]
            rows.append({
                "source": "packwise", "split": split,
                "stem": f"{split}_{Path(im.get('file_name', '')).stem}",
                "ann_id": int(ann["id"]), "image_id": int(ann["image_id"]),
                "item_id": f"packwise__{split}__ann{int(ann['id']):06d}",
                "class_name": name, "category_id": int(ann.get("category_id", -1)),
                "image_path": str(img_path),
                "img_w": int(im.get("width", 0)), "img_h": int(im.get("height", 0)),
                "bbox_x": x, "bbox_y": y, "bbox_w": bw, "bbox_h": bh,
                "coco_area": float(ann.get("area", np.nan)),
                "iscrowd": int(ann.get("iscrowd", 0)),
                "seg_type": ("rle" if isinstance(seg, dict)
                             else "polygon" if isinstance(seg, list) else "none"),
                "in_scope": bool(is_in_scope(name)),
            })
    INDEX["categories"] = cats_seen
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if df["item_id"].duplicated().any():
        dupes = df.loc[df["item_id"].duplicated(keep=False), "item_id"].unique()[:5]
        raise RuntimeError(f"duplicate item_id values, e.g. {list(dupes)}")
    return df.sort_values(["split", "stem", "ann_id"]).reset_index(drop=True)


# --- Widgets ---

w4_splits = widgets.SelectMultiple(options=list(SPLIT_FILES), value=tuple(SPLIT_FILES),
                                   description="Splits:", rows=3,
                                   style={"description_width": "130px"},
                                   layout=widgets.Layout(width="330px"))
w4_min_area = widgets.IntText(value=400, description="Min COCO area:",
                              style={"description_width": "130px"},
                              layout=widgets.Layout(width="260px"))
w4_drop_crowd = widgets.Checkbox(value=True, indent=False,
                                 description="Drop iscrowd=1 (RLE is a merged region)")
w4_nprev = widgets.IntSlider(value=6, min=1, max=12, description="Preview N:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="360px"))
w4_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "130px"},
                          layout=widgets.Layout(width="200px"))
w4_status = widgets.HTML()
w4_out = widgets.Output()


def _scope_objects():
    df = INDEX.get("objects")
    if df is None or df.empty:
        return pd.DataFrame()
    m = df["in_scope"] & df["split"].isin(list(w4_splits.value))
    m &= df["coco_area"].fillna(0) >= float(w4_min_area.value)
    if w4_drop_crowd.value:
        m &= df["iscrowd"] == 0
    return df[m]


def decode_ann_mask(ann, h, w):
    """COCO segmentation -> boolean mask. Handles RLE dicts and polygon lists."""
    if maskUtils is None:
        raise RuntimeError("pycocotools is required to decode PackWISE RLE masks")
    seg = ann.get("segmentation")
    if isinstance(seg, dict):
        rle = seg
        if isinstance(rle.get("counts"), list):        # uncompressed RLE
            rle = maskUtils.frPyObjects(rle, h, w)
        return maskUtils.decode(rle).astype(bool)
    if isinstance(seg, list):
        rles = maskUtils.frPyObjects(seg, h, w)
        return maskUtils.decode(maskUtils.merge(rles)).astype(bool)
    raise ValueError("annotation has no usable segmentation")


def render_preview4():
    with w4_out:
        clear_output(wait=True)
        df = build_object_index()
        INDEX["objects"] = df
        if df.empty:
            print("!! Index is empty - check DATASET_ROOT and the split JSONs.")
            return

        print("CATEGORY DECISIONS")
        print("-" * 78)
        counts = df.groupby("class_name").size().to_dict()
        dec = pd.DataFrame([
            {"category": k, "in_scope": v[0], "instances": counts.get(k, 0), "reason": v[1]}
            for k, v in CATEGORY_DECISIONS.items()
        ]).sort_values(["in_scope", "instances"], ascending=[False, False])
        display(HTML(dec.to_html(index=False)))
        undecided = sorted(set(df["class_name"]) - set(CATEGORY_DECISIONS))
        if undecided:
            print(f"!! categories present in the data with NO stated decision: {undecided}")
            print("!! add them to CATEGORY_DECISIONS before locking - silent exclusion is "
                  "the defect this table exists to prevent.")

        by_split = (df.groupby("split")
                      .agg(instances=("item_id", "size"), images=("stem", "nunique"),
                           in_scope=("in_scope", "sum"))
                      .reset_index())
        print("\nINDEX BY SPLIT")
        display(HTML(by_split.to_html(index=False)))
        print(f"segmentation encodings present: {df['seg_type'].value_counts().to_dict()}")
        print(f"iscrowd values present        : {sorted(df['iscrowd'].unique())}")

        scope = _scope_objects()
        print(f"\nin scope after split + area + crowd filter : {len(scope):,} instances "
              f"over {scope['stem'].nunique():,} images")
        if len(scope):
            display(HTML(scope.groupby(["split", "class_name"]).size()
                         .reset_index(name="instances")
                         .sort_values("instances", ascending=False).head(15)
                         .to_html(index=False)))

        # Overlay a few decoded masks to check RLE alignment with the image
        if len(scope):
            sample = scope.sample(n=min(w4_nprev.value, len(scope)),
                                  random_state=int(w4_seed.value))
            ann_cache = {}
            n = len(sample)
            ncol = 3
            nrow = int(np.ceil(n / ncol))
            fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 1.9, _FW * 0.62 * nrow),
                                     squeeze=False)
            axes = axes.reshape(-1)
            for ax, (_, r) in zip(axes, sample.iterrows()):
                try:
                    if r["split"] not in ann_cache:
                        d = _load_split(r["split"])
                        ann_cache[r["split"]] = {int(a["id"]): a for a in d["annotations"]}
                    ann = ann_cache[r["split"]][int(r["ann_id"])]
                    m = decode_ann_mask(ann, int(r["img_h"]), int(r["img_w"]))
                    rgb = load_rgb(r["image_path"])
                except Exception as e:
                    ax.text(0.5, 0.5, f"failed\n{type(e).__name__}: {e}", ha="center",
                            fontsize=7)
                    ax.axis("off")
                    continue
                # Downsample for display
                step = max(1, int(max(rgb.shape[:2]) / 900))
                ax.imshow(rgb[::step, ::step])
                ax.contour(m[::step, ::step], levels=[0.5], colors="#FF0000", linewidths=1.0)
                ax.set_title(f"{r['class_name']}  {r['split']}\n"
                             f"mask {int(m.sum()):,} px  coco_area {r['coco_area']:.0f}",
                             fontsize=7)
                ax.axis("off")
            for ax in axes[n:]:
                ax.axis("off")
            _titled_legend(fig, "Index Preview - Decoded COCO Masks on In-Scope Instances",
                           top_rect=0.92)
            plt.show()

        w4_status.value = status_html("index", n_scope=len(scope))


def lock4():
    params = {"splits": list(w4_splits.value),
              "min_coco_area": float(w4_min_area.value),
              "drop_iscrowd": bool(w4_drop_crowd.value),
              "in_scope_criterion": IN_SCOPE_CRITERION,
              "category_decisions": {k: {"in_scope": v[0], "reason": v[1]}
                                     for k, v in CATEGORY_DECISIONS.items()}}
    lock_stage("index", params, n_preview=int(w4_nprev.value))
    w4_status.value = status_html("index", n_scope=len(_scope_objects()))


def batch4():
    p = require_locked("index")
    with w4_out:
        clear_output(wait=True)
        df = INDEX.get("objects")
        if df is None or df.empty:
            df = build_object_index()
            INDEX["objects"] = df
        scope = _scope_objects()
        out_all = DIR_TABLES / "object_index.csv"
        out_scope = DIR_TABLES / "object_index_in_scope.csv"
        df.to_csv(out_all, index=False)
        scope.to_csv(out_scope, index=False)
        log_params("index",
                   outputs={"object_index": str(out_all),
                            "object_index_in_scope": str(out_scope)},
                   results={"n_instances_total": int(len(df)),
                            "n_instances_in_scope": int(len(scope)),
                            "n_images_in_scope": int(scope["stem"].nunique()),
                            "by_split": scope.groupby("split").size().to_dict()})
        print(f"index    -> {out_all}   ({len(df):,} rows)")
        print(f"in-scope -> {out_scope}   ({len(scope):,} rows)")
        w4_status.value = status_html("index", n_scope=len(scope))


w4_bar, w4_buttons = make_action_bar(render_preview4, lock4, batch4,
                                     batch_label="Write index CSVs")
for _w in (w4_splits, w4_min_area, w4_drop_crowd):
    _w.observe(lambda ch: invalidate_lock("index", w4_buttons), names="value")

w4_status.value = status_html("index")
display(widgets.VBox([
    widgets.HBox([w4_splits, w4_min_area]),
    widgets.HBox([w4_drop_crowd]),
    widgets.HBox([w4_nprev, w4_seed]),
    w4_bar,
    w4_status,
]), w4_out)

In [ ]:
# === Cell 6: SAM3 Runtime and Worker Dispatch ===
# SAM3 is used only for Stage 3 (cap/label prompts); whole-object masks come from the COCO
# RLE annotations. Workers run as a standalone .py via Popen with sys.executable, one shard
# file and one log file per worker.

SAM3_WORKER_SRC = r'''#!/usr/bin/env python
"""Generated by 20260820_PackWISE_Preprocessing.ipynb - do not edit by hand.
This copy is an artifact of the run that produced it."""
import argparse, json, sys, time
from pathlib import Path
import numpy as np
from PIL import Image

_state = {"pred": None, "last_image": None, "imgsz": None}


def get_predictor(weights, imgsz, conf):
    if _state["pred"] is None or _state["imgsz"] != imgsz:
        from ultralytics.models.sam import SAM3SemanticPredictor
        overrides = {"conf": conf, "task": "segment", "mode": "predict",
                     "model": str(weights), "imgsz": int(imgsz),
                     "save": False, "verbose": False}
        _state["pred"] = SAM3SemanticPredictor(overrides=overrides)
        _state["imgsz"] = imgsz
        _state["last_image"] = None
    return _state["pred"]


def run_sam3(image_path, prompts, conf, weights, imgsz):
    """Encoder cache: set_image dominates runtime, decode is cheap. Only re-encode
    when the image actually changes, so multi-prompt on one frame is near-free."""
    pred = get_predictor(weights, imgsz, conf)
    pred.args.conf = float(conf)
    if _state["last_image"] != str(image_path):
        pred.set_image(str(image_path))
        _state["last_image"] = str(image_path)
    return pred(text=list(prompts))[0]


def masks_to_bool(result):
    """(N, H, W) bool at ORIGINAL image resolution."""
    if result.masks is None or len(result.masks) == 0:
        return np.zeros((0, *result.orig_shape), dtype=bool)
    data = result.masks.data.cpu().numpy()
    H, W = result.orig_shape
    if data.shape[1:] != (H, W):
        out = np.zeros((data.shape[0], H, W), dtype=bool)
        for i, m in enumerate(data):
            im = Image.fromarray((m > 0.5).astype(np.uint8) * 255)
            out[i] = np.asarray(im.resize((W, H), Image.NEAREST)) > 127
        return out
    return data > 0.5


def _lookup_name(names, cls_id):
    """result.names is usually {cls_id: name}, but some SAM3 predictors return it as a
    plain list indexed by cls_id - .get() doesn't exist on a list, hence the crash."""
    if isinstance(names, dict):
        return str(names.get(cls_id, cls_id))
    try:
        return str(names[cls_id])
    except (IndexError, TypeError):
        return str(cls_id)


def det_meta(result):
    if result.boxes is None or len(result.boxes) == 0:
        return [], [], []
    confs = [float(c) for c in result.boxes.conf.cpu().numpy()]
    clsid = [int(c) for c in result.boxes.cls.cpu().numpy()]
    names = [_lookup_name(result.names, c) for c in clsid]
    return confs, clsid, names


def save_npz(path, provenance=None, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(arrays)
    if provenance is not None:
        payload["provenance"] = np.array(json.dumps(provenance, default=str))
    np.savez_compressed(path, **payload)


# ------------------------------------------------------------------ stage 3
def do_stage3(items, p, mask_dir, writer):
    """cap/label sub-segmentation on cutouts -> body/cap/label pseudo-GT.

    Measures only. Every diagnostic (n_det, conf, area fraction, per region) is written
    as-is; NOTHING is gated here. All thresholds live in the notebook's filter cell so
    re-tuning them costs milliseconds instead of a SAM3 re-run.

    whole_object_mask is REPLAYED from the Stage-1b mask through the canvas geometry,
    never derived from cutout pixel colours (dark object regions would read as
    background).
    """
    weights, imgsz = p["weights"], p["imgsz"]
    cap_prompt, label_prompt = p["cap_prompt"], p["label_prompt"]
    # cap and label get their OWN confidence threshold - the two prompts detect very
    # different-looking regions (a whole cap vs. a printed label) and a single shared
    # conf was never a real constraint, just an accident of one widget doing two jobs.
    conf_map = {"cap": p["cap_conf"], "label": p["label_conf"]}
    for it in items:
        rec = {"item_id": it["cutout_id"], "cutout_id": it["cutout_id"],
               "stem": it.get("stem", ""), "cutout_path": it["cutout_path"], "error": ""}
        try:
            with np.load(it["whole_mask_path"], allow_pickle=False) as z:
                src_mask = z["mask"]
            ch, cw = int(it["canvas_h"]), int(it["canvas_w"])
            whole = np.zeros((ch, cw), dtype=bool)
            ys, xs = np.nonzero(src_mask)
            cy = ys - int(it["mask_bbox_y1"]) + int(it["canvas_offset_y"])
            cx = xs - int(it["mask_bbox_x1"]) + int(it["canvas_offset_x"])
            ok = (cy >= 0) & (cy < ch) & (cx >= 0) & (cx < cw)
            whole[cy[ok], cx[ok]] = True
            whole_area = int(whole.sum())
            rec["whole_area_px"] = whole_area
            if whole_area == 0:
                rec["error"] = "empty whole-object mask"
                writer(rec)
                continue

            region = {}
            for key, prompt in (("cap", cap_prompt), ("label", label_prompt)):
                try:
                    res = run_sam3(it["cutout_path"], [prompt], conf_map[key], weights, imgsz)
                    m = masks_to_bool(res)
                    confs, _, _ = det_meta(res)
                    union = m.any(axis=0) if m.shape[0] else np.zeros_like(whole)
                    if union.shape != whole.shape:
                        im = Image.fromarray(union.astype(np.uint8) * 255)
                        union = np.asarray(im.resize((cw, ch), Image.NEAREST)) > 127
                    union &= whole
                    region[key] = union
                    rec[key + "_n_det"] = int(m.shape[0])
                    rec[key + "_conf"] = float(max(confs)) if confs else np.nan
                    rec[key + "_frac"] = float(union.sum()) / whole_area
                except Exception as e:
                    region[key] = np.zeros_like(whole)
                    rec[key + "_n_det"] = 0
                    rec[key + "_conf"] = np.nan
                    rec[key + "_frac"] = 0.0
                    rec[key + "_error"] = f"{type(e).__name__}: {e}"

            body = whole & ~region["cap"] & ~region["label"]
            rec["body_frac"] = float(body.sum()) / whole_area
            rec["cap_present"] = bool(region["cap"].any())

            prov = {"cutout_id": it["cutout_id"], "source_cutout": it["cutout_path"],
                    "source_whole_mask": it["whole_mask_path"],
                    "cap_prompt": cap_prompt, "label_prompt": label_prompt,
                    "cap_conf": conf_map["cap"], "label_conf": conf_map["label"],
                    "imgsz": imgsz, "run_id": p.get("run_id")}
            mp = Path(mask_dir) / f"{it['cutout_id']}_regions.npz"
            save_npz(mp, body=body, cap=region["cap"], label=region["label"],
                     whole=whole, provenance=prov)
            rec["regions_mask_path"] = str(mp)
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
        writer(rec)


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", required=True, choices=["stage3"])
    ap.add_argument("--items", required=True)
    ap.add_argument("--params", required=True)
    ap.add_argument("--out-csv", required=True)
    ap.add_argument("--mask-dir", required=True)
    args = ap.parse_args()

    items = json.loads(Path(args.items).read_text())
    p = json.loads(Path(args.params).read_text())
    Path(args.mask_dir).mkdir(parents=True, exist_ok=True)
    out = Path(args.out_csv)
    out.parent.mkdir(parents=True, exist_ok=True)

    import csv
    # FIXED per-stage header, not inferred from whichever row happens to be written first.
    # do_stage3 emits a "thin" row (no regions_mask_path at all) whenever an object's
    # whole-mask replay is empty; if that thin row landed first, a keys-of-first-row header
    # plus extrasaction="ignore" would silently drop regions_mask_path/cap_frac/label_frac
    # from every later row - even fully successful ones.
    STAGE_FIELDS = {
        "stage3": ["item_id", "cutout_id", "stem", "cutout_path", "whole_area_px",
                   "cap_n_det", "cap_conf", "cap_frac", "cap_error",
                   "label_n_det", "label_conf", "label_frac", "label_error",
                   "body_frac", "cap_present", "regions_mask_path", "error"],
    }
    state = {"fh": None, "w": None}

    def writer(row):
        if state["w"] is None:
            state["fh"] = open(out, "w", newline="", buffering=1)
            state["w"] = csv.DictWriter(state["fh"], fieldnames=STAGE_FIELDS[args.stage],
                                        extrasaction="ignore", restval="")
            state["w"].writeheader()
        state["w"].writerow(row)
        state["fh"].flush()

    t0 = time.time()
    print(f"[worker] stage={args.stage} items={len(items)} device_env="
          f"{__import__('os').environ.get('CUDA_VISIBLE_DEVICES', 'unset')}", flush=True)
    do_stage3(items, p, args.mask_dir, writer)
    if state["fh"]:
        state["fh"].close()
    print(f"[worker] done in {time.time() - t0:.1f}s", flush=True)


if __name__ == "__main__":
    main()
'''


def write_worker_script():
    """Write the worker script into the run directory."""
    p = DIR_WORKERS / "sam3_worker.py"
    p.write_text(SAM3_WORKER_SRC)
    p.chmod(0o755)
    return p


WORKER_PATH = write_worker_script()


# --- In-process SAM3 (preview only) ---

_PREVIEW = {"pred": None, "last_image": None, "imgsz": None}


def preview_predictor(imgsz, conf):
    if _PREVIEW["pred"] is None or _PREVIEW["imgsz"] != imgsz:
        from ultralytics.models.sam import SAM3SemanticPredictor
        overrides = {"conf": conf, "task": "segment", "mode": "predict",
                     "model": str(SAM3_WEIGHTS), "imgsz": int(imgsz),
                     "save": False, "verbose": False}
        _PREVIEW["pred"] = SAM3SemanticPredictor(overrides=overrides)
        _PREVIEW["imgsz"] = imgsz
        _PREVIEW["last_image"] = None
    return _PREVIEW["pred"]


def sam3_preview(image_path, prompts, conf, imgsz):
    pred = preview_predictor(imgsz, conf)
    pred.args.conf = float(conf)
    if _PREVIEW["last_image"] != str(image_path):
        pred.set_image(str(image_path))
        _PREVIEW["last_image"] = str(image_path)
    return pred(text=list(prompts))[0]


def release_preview_model():
    """Free the notebook's SAM3 model and CUDA cache before a batch run."""
    _PREVIEW["pred"] = None
    _PREVIEW["last_image"] = None
    _PREVIEW["imgsz"] = None
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


def result_masks_bool(result):
    if result.masks is None or len(result.masks) == 0:
        return np.zeros((0, *result.orig_shape), dtype=bool)
    data = result.masks.data.cpu().numpy()
    H, W = result.orig_shape
    if data.shape[1:] != (H, W):
        out = np.zeros((data.shape[0], H, W), dtype=bool)
        for i, m in enumerate(data):
            im = Image.fromarray((m > 0.5).astype(np.uint8) * 255)
            out[i] = np.asarray(im.resize((W, H), Image.NEAREST)) > 127
        return out
    return data > 0.5


def _lookup_name(names, cls_id):
    """Class name for cls_id from a dict or list; mirrors the helper in SAM3_WORKER_SRC."""
    if isinstance(names, dict):
        return str(names.get(cls_id, cls_id))
    try:
        return str(names[cls_id])
    except (IndexError, TypeError):
        return str(cls_id)


def result_meta(result):
    if result.boxes is None or len(result.boxes) == 0:
        return [], []
    confs = [float(c) for c in result.boxes.conf.cpu().numpy()]
    names = [_lookup_name(result.names, int(c)) for c in result.boxes.cls.cpu().numpy()]
    return confs, names


# --- Dispatch ---

def run_sam3_batch(stage, items, params, mask_dir, key_col="item_id"):
    """Shard `items` across available GPUs, run the worker, aggregate, report ETA.

    Items are sharded round-robin. Progress is polled as the number of distinct
    completed keys across shard CSVs.
    """
    plan = resolve_compute_plan()
    n_workers = plan["n_workers"] if plan["mode"] == "multi" else 1
    devices = plan["devices"] or [None]

    _, done = load_checkpoint(stage, key_col=key_col)
    todo = [it for it in items if str(it.get(key_col, it.get("stem"))) not in done]
    if done:
        print(f"[{stage}] resume: {len(done):,} already done, {len(todo):,} remaining")
    if not todo:
        print(f"[{stage}] nothing to do.")
        return pd.read_csv(checkpoint_path(stage)) if checkpoint_path(stage).exists() else pd.DataFrame()

    release_preview_model()
    Path(mask_dir).mkdir(parents=True, exist_ok=True)
    shard_dir = DIR_WORKERS / f"{stage}_shards"
    shard_dir.mkdir(parents=True, exist_ok=True)

    params = dict(params)
    params.setdefault("run_id", RUN_ID)
    params["weights"] = str(SAM3_WEIGHTS)
    params_path = shard_dir / "params.json"
    params_path.write_text(json.dumps(params, indent=2, default=str))

    shards = [todo[i::n_workers] for i in range(n_workers)]
    procs, shard_csvs, logs = [], [], []
    for wi, shard in enumerate(shards):
        if not shard:
            continue
        sp = shard_dir / f"shard_{wi:02d}.json"          # one shard file per worker
        sp.write_text(json.dumps(shard, default=str))
        oc = shard_dir / f"out_{wi:02d}.csv"
        lg = DIR_LOGS / f"{stage}_worker{wi:02d}.log"
        shard_csvs.append(oc)
        logs.append(lg)

        env = dict(os.environ)
        dev = devices[wi % len(devices)]
        if dev is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(dev)
        cmd = [sys.executable, str(WORKER_PATH),          # use the kernel's interpreter
               "--stage", stage, "--items", str(sp), "--params", str(params_path),
               "--out-csv", str(oc), "--mask-dir", str(mask_dir)]
        fh = open(lg, "w", buffering=1)
        procs.append((wi, subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env), fh, oc))
        print(f"[{stage}] worker {wi} -> GPU {dev}  {len(shard):,} items  log={lg.name}")

    rep = ProgressReporter(stage, len(todo))
    try:
        while any(pr.poll() is None for _, pr, _, _ in procs):
            time.sleep(WORKER_POLL_S)
            rep.set_done(sum(count_checkpoint_items(c, key_col=key_col) for c in shard_csvs))
            rep.report()
    finally:
        for wi, pr, fh, _ in procs:
            pr.wait()
            fh.close()

    failed = [(wi, pr.returncode, logs[i]) for i, (wi, pr, _, _) in enumerate(procs)
              if pr.returncode != 0]
    for wi, rc, lg in failed:
        print(f"!! worker {wi} exited {rc} - see {lg}")
        try:
            for line in lg.read_text().strip().splitlines()[-15:]:
                print(f"     {line}")
        except OSError:
            pass

    frames = [pd.read_csv(c) for c in shard_csvs if c.exists() and c.stat().st_size > 0]
    new = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if len(new):
        append_checkpoint(stage, new.to_dict("records"))
    rep.set_done(len(new), force=True)
    rep.close(extra={"n_workers": len(procs), "mode": plan["mode"],
                     "devices": plan["devices"], "failed_workers": len(failed)})

    ck = checkpoint_path(stage)
    return pd.read_csv(ck) if ck.exists() else new


print(f"SAM3 worker written -> {WORKER_PATH}")
print(f"dispatch mode       : {COMPUTE_PLAN['mode']} ({COMPUTE_PLAN['n_workers']} worker(s))")

In [ ]:
# === Cell 7: Stage 1b: Whole-Object Masks ===
# CPU-only decode of the annotated COCO RLE masks (no model). Work is grouped by split and
# image so each split JSON is parsed once. Masks are saved at full resolution as .npz.

STAGE1B = {"masks": None, "sec_per_item": None}


def _ann_lookup(split):
    """ann_id -> annotation dict, for one split. Cached by the caller."""
    d = _load_split(split)
    if d is None:
        return {}
    return {int(a["id"]): a for a in d.get("annotations", [])}


w6_minarea = widgets.IntText(value=200, description="Min mask px:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="240px"))
w6_nprev = widgets.IntSlider(value=6, min=1, max=12, description="Preview N:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="360px"))
w6_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "130px"},
                          layout=widgets.Layout(width="200px"))
w6_status = widgets.HTML()
w6_out = widgets.Output()


def render_preview6():
    with w6_out:
        clear_output(wait=True)
        scope = _scope_objects()
        if scope.empty:
            print("!! Nothing in scope - run Cell 5 first.")
            return
        if maskUtils is None:
            print("!! pycocotools is not installed; this stage cannot run.")
            print("!! install with:  pip install pycocotools")
            return
        print(f"in scope: {len(scope):,} instances over {scope['stem'].nunique():,} images")
        print("decode is CPU-only and needs no model - the whole-object mask is ground "
              "truth, not an estimate")

        sample = scope.sample(n=min(w6_nprev.value, len(scope)), random_state=int(w6_seed.value))
        ann_cache, times, rows = {}, [], []
        n = len(sample)
        ncol = 3
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 1.9, _FW * 0.62 * nrow),
                                 squeeze=False)
        axes = axes.reshape(-1)
        for ax, (_, r) in zip(axes, sample.iterrows()):
            t0 = time.time()
            try:
                if r["split"] not in ann_cache:
                    ann_cache[r["split"]] = _ann_lookup(r["split"])
                ann = ann_cache[r["split"]][int(r["ann_id"])]
                m = decode_ann_mask(ann, int(r["img_h"]), int(r["img_w"]))
                rgb = load_rgb(r["image_path"])
            except Exception as e:
                ax.text(0.5, 0.5, f"failed\n{type(e).__name__}: {e}", ha="center", fontsize=7)
                ax.axis("off")
                continue
            times.append(time.time() - t0)
            area = int(m.sum())
            step = max(1, int(max(rgb.shape[:2]) / 900))
            ax.imshow(rgb[::step, ::step])
            ax.contour(m[::step, ::step], levels=[0.5], colors="#FF0000", linewidths=1.0)
            ax.set_title(f"{r['class_name']}  {r['split']}\n"
                         f"{area:,} px  ({100 * area / (r['img_w'] * r['img_h']):.2f}% of frame)",
                         fontsize=7)
            ax.axis("off")
            rows.append({"class": r["class_name"], "mask_px": area,
                         "coco_area": r["coco_area"],
                         "ratio": round(area / max(r["coco_area"], 1), 3)})
        for ax in axes[n:]:
            ax.axis("off")
        _titled_legend(fig, "Stage 1b Preview - Decoded Ground-Truth Instance Masks",
                       top_rect=0.92)
        plt.show()

        if rows:
            d = pd.DataFrame(rows)
            display(HTML(d.to_html(index=False)))
            # Decoded area should match the annotation's area field (within 5%)
            bad = d[(d["ratio"] < 0.95) | (d["ratio"] > 1.05)]
            if len(bad):
                print(f"!! {len(bad)} of {len(d)} masks disagree with coco_area by >5% - "
                      f"check the height/width order passed to the decoder.")
            else:
                print("decoded mask areas agree with the annotation area field (within 5%)")

        spi = float(np.median(times)) if times else None
        STAGE1B["sec_per_item"] = spi
        est = estimate_batch_runtime(spi, len(scope))
        print(f"\nmedian {spi:.2f} s/instance (includes full-res JPEG decode) -> "
              f"projected batch over {len(scope):,}: {est}" if spi else "")
        print("(the batch groups by image, so the real cost is well below this estimate)")
        w6_status.value = status_html("stage1b", n_scope=len(scope), est=est)


def lock6():
    params = {"min_mask_px": int(w6_minarea.value), "mode": "coco_rle_decode"}
    lock_stage("stage1b", params, n_preview=int(w6_nprev.value))
    w6_status.value = status_html("stage1b", n_scope=len(_scope_objects()))


def batch6():
    p = require_locked("stage1b")
    scope = _scope_objects()
    with w6_out:
        clear_output(wait=True)
        if scope.empty:
            print("!! Nothing in scope.")
            return
        if maskUtils is None:
            print("!! pycocotools is not installed; aborting.")
            return
        mask_dir = DIR_STAGE1 / "masks"
        mask_dir.mkdir(parents=True, exist_ok=True)
        print(f"Stage 1b: decoding {len(scope):,} RLE masks "
              f"over {scope['stem'].nunique():,} images")

        rows = []
        ann_cache = {}
        rep = ProgressReporter("stage1b", len(scope))
        # Sorted by split, then image
        for _, r in scope.sort_values(["split", "stem"]).iterrows():
            rec = {"item_id": r["item_id"], "stem": r["stem"], "source": "packwise",
                   "split": r["split"], "class_name": r["class_name"],
                   "ann_id": int(r["ann_id"]), "image_path": r["image_path"], "error": ""}
            try:
                if r["split"] not in ann_cache:
                    ann_cache[r["split"]] = _ann_lookup(r["split"])
                ann = ann_cache[r["split"]][int(r["ann_id"])]
                m = decode_ann_mask(ann, int(r["img_h"]), int(r["img_w"]))
                area = int(m.sum())
                if area < int(p["min_mask_px"]):
                    rec["mask_area_px"] = area
                    rec["error"] = f"mask below min_mask_px ({area} < {p['min_mask_px']})"
                    rows.append(rec)
                    rep.update()
                    continue
                ys, xs = np.nonzero(m)
                mp = mask_dir / f"{r['item_id']}.npz"
                save_mask_npz(mp, mask=m,
                              provenance=provenance_record(
                                  "stage1b", {"rgb": r["image_path"],
                                              "coco_json": str(SPLIT_FILES[r["split"]])},
                                  p, extra={"item_id": r["item_id"],
                                            "ann_id": int(r["ann_id"]),
                                            "class_name": r["class_name"],
                                            "mode": "coco_rle_decode"}))
                rec.update({"mask_area_px": area,
                            "x1": int(xs.min()), "y1": int(ys.min()),
                            "x2": int(xs.max()), "y2": int(ys.max()),
                            "img_w": int(m.shape[1]), "img_h": int(m.shape[0]),
                            "coco_area": float(r["coco_area"]),
                            "area_ratio_to_coco": area / max(float(r["coco_area"]), 1.0),
                            "whole_mask_path": str(mp)})
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"
            rows.append(rec)
            rep.update()
        rep.close()

        df = pd.DataFrame(rows)
        df["error"] = df["error"].fillna("")
        ok = df[(df["error"] == "") & df["whole_mask_path"].notna()]
        out_csv = DIR_STAGE1 / "whole_masks.csv"
        df.to_csv(out_csv, index=False)
        STAGE1B["masks"] = ok

        print(f"\ninstances processed : {len(df):,}")
        print(f"masks produced      : {len(ok):,}  ({100 * len(ok) / max(len(df), 1):.1f}%)")
        n_err = int((df["error"] != "").sum())
        print(f"errors / rejects    : {n_err:,}")
        if n_err:
            display(HTML(df[df["error"] != ""]["error"].str.slice(0, 60)
                         .value_counts().head(10).to_frame("n").to_html()))
        if len(ok):
            display(HTML(ok.groupby(["split", "class_name"])
                         .agg(n=("item_id", "size"),
                              median_area=("mask_area_px", "median"),
                              median_ratio=("area_ratio_to_coco", "median"))
                         .reset_index().to_html(index=False)))
            off = ok[(ok["area_ratio_to_coco"] < 0.95) | (ok["area_ratio_to_coco"] > 1.05)]
            print(f"masks disagreeing with coco_area by >5%: {len(off):,} "
                  f"({100 * len(off) / max(len(ok), 1):.1f}%)")
        log_params("stage1b", outputs={"whole_masks": str(out_csv), "masks": str(mask_dir)},
                   results={"n_instances": int(len(df)), "n_masks": int(len(ok)),
                            "n_errors": n_err})
        print(f"\nwhole masks -> {out_csv}")
        w6_status.value = status_html("stage1b", n_scope=len(scope))


w6_bar, w6_buttons = make_action_bar(render_preview6, lock6, batch6,
                                     batch_label="Run Stage 1b decode")
w6_minarea.observe(lambda ch: invalidate_lock("stage1b", w6_buttons), names="value")

w6_status.value = status_html("stage1b")
display(widgets.VBox([
    widgets.HBox([w6_minarea]),
    widgets.HBox([w6_nprev, w6_seed]),
    w6_bar,
    w6_status,
]), w6_out)

In [ ]:
# === Cell 8: Stage 2: Black-Canvas Cutouts ===
# CPU-only. Mask-isolated object on a black canvas at the 640:384 inference aspect.
# `make_cutout` matches the other preprocessing notebooks, including its exclusive bbox
# slice (last row/column dropped), so cutouts stay comparable across datasets.
# Instances are processed in image order with a one-frame cache. The coordinate map
# (mask_bbox_*, canvas_offset_*) maps cutouts back to the frame and is used by Stage 3.

STAGE2 = {"cutouts": None, "sec_per_item": None}


def make_cutout(rgb_arr, mask, aspect_w, aspect_h, canvas_scale):
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None, None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    obj_w, obj_h = max(x2 - x1, 1), max(y2 - y1, 1)

    tight = np.zeros((obj_h, obj_w, 3), dtype=np.uint8)
    local = mask[y1:y2, x1:x2]
    tight[local] = rgb_arr[y1:y2, x1:x2][local]

    target_ratio = aspect_w / aspect_h
    canvas_h = max(obj_h, obj_w / target_ratio) * canvas_scale
    canvas_w = canvas_h * target_ratio
    canvas_h, canvas_w = int(canvas_h), int(canvas_w)

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    off_x, off_y = (canvas_w - obj_w) // 2, (canvas_h - obj_h) // 2
    canvas[off_y:off_y + obj_h, off_x:off_x + obj_w] = tight

    coord_map = {"mask_bbox_x1": x1, "mask_bbox_y1": y1,
                 "mask_bbox_x2": x2, "mask_bbox_y2": y2,
                 "canvas_offset_x": off_x, "canvas_offset_y": off_y,
                 "canvas_w": canvas_w, "canvas_h": canvas_h,
                 "obj_w": obj_w, "obj_h": obj_h,
                 "fill_frac": float(mask.sum()) / (canvas_w * canvas_h)}
    return canvas, coord_map


class _FrameCache:
    """Keeps the last decoded frame (instances are iterated in image order)."""

    def __init__(self):
        self.path = None
        self.arr = None
        self.n_decodes = 0

    def get(self, path):
        if path != self.path:
            self.arr = load_rgb(path)
            self.path = path
            self.n_decodes += 1
        return self.arr


def _whole_masks_df():
    df = STAGE1B.get("masks")
    if df is None:
        p = DIR_STAGE1 / "whole_masks.csv"
        if p.exists():
            df = pd.read_csv(p)
            df["error"] = df["error"].fillna("")
            df = df[(df["error"] == "") & df["whole_mask_path"].notna()]
            STAGE1B["masks"] = df
    return df if df is not None else pd.DataFrame()


# --- Widgets ---

w7_scale = widgets.FloatSlider(value=2.0, min=1.0, max=6.0, step=0.1,
                               description="Canvas scale:", readout_format=".1f",
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="420px"))
w7_aspect = widgets.HTML(
    f"<div style='font-family:monospace;font-size:12px;padding:6px'>"
    f"canvas aspect fixed at <b>{CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}</b> "
    f"(deployment model inference shape {YOLO_INFER_H}x{YOLO_INFER_W})</div>")
w7_nprev = widgets.IntSlider(value=6, min=1, max=12, description="Preview N:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="360px"))
w7_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "130px"},
                          layout=widgets.Layout(width="200px"))
w7_status = widgets.HTML()
w7_out = widgets.Output()


def render_preview7():
    with w7_out:
        clear_output(wait=True)
        wm = _whole_masks_df()
        if wm.empty:
            print("!! No whole-object masks - run Cell 7 first.")
            return
        sample = wm.sample(n=min(w7_nprev.value, len(wm)), random_state=int(w7_seed.value))
        cache = _FrameCache()
        n = len(sample)
        ncol = 3
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 1.9, _FW * 0.66 * nrow),
                                 squeeze=False)
        axes = axes.reshape(-1)
        t_items, fills = [], []
        for ax, (_, r) in zip(axes, sample.sort_values("image_path").iterrows()):
            t0 = time.time()
            try:
                arrays, _ = read_mask_npz(r["whole_mask_path"])
                mask = arrays["mask"].astype(bool)
                rgb = cache.get(r["image_path"])
                canvas, cmap = make_cutout(rgb, mask, CANVAS_ASPECT_W, CANVAS_ASPECT_H,
                                           float(w7_scale.value))
            except Exception as e:
                ax.text(0.5, 0.5, f"failed\n{e}", ha="center", fontsize=7)
                ax.axis("off")
                continue
            t_items.append(time.time() - t0)
            if canvas is None:
                ax.text(0.5, 0.5, "empty mask", ha="center", fontsize=8)
                ax.axis("off")
                continue
            fills.append(cmap["fill_frac"])
            ax.imshow(canvas)
            ax.set_title(f"{r['class_name']}  {r['split']}\n"
                         f"{cmap['canvas_w']}x{cmap['canvas_h']}  "
                         f"fill={cmap['fill_frac']:.3f}", fontsize=7)
            ax.axis("off")
        for ax in axes[n:]:
            ax.axis("off")
        _titled_legend(fig, f"Stage 2 Preview - Black-Canvas Cutouts  "
                            f"aspect {CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}  "
                            f"scale={w7_scale.value:.1f}", top_rect=0.92)
        plt.show()

        if fills:
            print(f"object fill fraction of canvas: median {np.median(fills):.4f}  "
                  f"min {min(fills):.4f}  max {max(fills):.4f}")
            print("(KI-Sort training reference, fraction of frame: body median 0.0251, "
                  "cap 0.0069, label 0.0423)")
        print(f"frames decoded for this preview: {cache.n_decodes} for {n} instances")
        spi = float(np.median(t_items)) if t_items else None
        STAGE2["sec_per_item"] = spi
        est = estimate_batch_runtime(spi, len(wm))
        print(f"\nmedian {spi:.2f} s/item -> projected batch over {len(wm):,} items: {est}"
              if spi else "")
        w7_status.value = status_html("stage2", n_scope=len(wm), est=est)


def lock7():
    params = {"aspect_w": int(CANVAS_ASPECT_W), "aspect_h": int(CANVAS_ASPECT_H),
              "canvas_scale": float(w7_scale.value), "cutout_format": CUTOUT_FORMAT}
    lock_stage("stage2", params, n_preview=int(w7_nprev.value))
    w7_status.value = status_html("stage2", n_scope=len(_whole_masks_df()))


def batch7():
    p = require_locked("stage2")
    wm = _whole_masks_df()
    with w7_out:
        clear_output(wait=True)
        if wm.empty:
            print("!! No whole-object masks.")
            return
        cut_dir = DIR_CUTOUTS / "cutouts"
        cut_dir.mkdir(parents=True, exist_ok=True)
        print(f"Stage 2: {len(wm):,} cutouts | aspect {p['aspect_w']}:{p['aspect_h']} "
              f"scale={p['canvas_scale']}")

        rows, prov_rows = [], []
        cache = _FrameCache()
        rep = ProgressReporter("stage2", len(wm))
        # Image order keeps the one-frame cache effective
        for _, r in wm.sort_values(["image_path", "item_id"]).iterrows():
            rec = {**{k: r[k] for k in ("item_id", "stem", "source", "split", "class_name",
                                        "image_path", "whole_mask_path")},
                   "cutout_id": r["item_id"], "error": ""}
            try:
                arrays, _ = read_mask_npz(r["whole_mask_path"])
                mask = arrays["mask"].astype(bool)
                rgb = cache.get(r["image_path"])
                canvas, cmap = make_cutout(rgb, mask, p["aspect_w"], p["aspect_h"],
                                           p["canvas_scale"])
                if canvas is None:
                    rec["error"] = "empty mask"
                else:
                    out_path = cut_dir / f"{r['item_id']}.{CUTOUT_FORMAT}"
                    prov = provenance_record(
                        "stage2", {"rgb": r["image_path"], "whole_mask": r["whole_mask_path"]},
                        p, extra={"item_id": r["item_id"], "class_name": r["class_name"],
                                  "split": r["split"], "source": "packwise"})
                    save_image_with_provenance(out_path, canvas, provenance=prov)
                    rec.update(cmap)
                    rec["cutout_path"] = str(out_path)
                    prov_rows.append({**prov, "cutout_path": str(out_path)})
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"
            rows.append(rec)
            rep.update()
        rep.close()

        df = pd.DataFrame(rows)
        out_csv = DIR_CUTOUTS / "cutouts.csv"
        df.to_csv(out_csv, index=False)
        write_provenance_csv("stage2", prov_rows)
        ok = df[(df["error"] == "") & df.get("cutout_path", pd.Series(dtype=object)).notna()]
        STAGE2["cutouts"] = ok

        print(f"\ncutouts written : {len(ok):,}/{len(df):,}")
        n_err = int((df["error"] != "").sum())
        print(f"errors          : {n_err:,}")
        print(f"frames decoded  : {cache.n_decodes:,} for {len(df):,} instances "
              f"({len(df) / max(cache.n_decodes, 1):.1f} instances per decode)")
        if len(ok):
            print(f"median canvas   : {int(ok['canvas_w'].median())}x{int(ok['canvas_h'].median())}")
            print(f"median fill_frac: {ok['fill_frac'].median():.4f}")
        log_params("stage2", outputs={"cutouts_csv": str(out_csv), "cutouts": str(cut_dir)},
                   results={"n_in": int(len(df)), "n_cutouts": int(len(ok)),
                            "n_errors": n_err, "n_frame_decodes": int(cache.n_decodes)})
        print(f"\ncutouts -> {out_csv}")
        w7_status.value = status_html("stage2", n_scope=len(wm))


w7_bar, w7_buttons = make_action_bar(render_preview7, lock7, batch7,
                                     batch_label="Run Stage 2 batch")
w7_scale.observe(lambda ch: invalidate_lock("stage2", w7_buttons), names="value")

w7_status.value = status_html("stage2")
display(widgets.VBox([
    w7_aspect,
    widgets.HBox([w7_scale]),
    widgets.HBox([w7_nprev, w7_seed]),
    w7_bar,
    w7_status,
]), w7_out)

In [ ]:
# === Cell 9: Stage 3: SAM3 Cap/Label Pseudo-GT ===
# Runs SAM3 cap/label prompts on cutouts; body = whole - cap - label (cutout coordinates).
# Raw diagnostics are stored unfiltered; all thresholds are applied in the Pseudo-GT Filter
# cell. The whole mask is replayed from Stage 1b, not derived from cutout pixel colours.

STAGE3 = {"pseudo_gt": None, "sec_per_item": None}

w8_cap_prompt = widgets.Text(value="cap", description="Cap prompt:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="380px"))
w8_label_prompt = widgets.Text(value="label", description="Label prompt:",
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="380px"))
w8_cap_conf = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                                  description="Cap conf:", readout_format=".2f",
                                  style={"description_width": "130px"},
                                  layout=widgets.Layout(width="380px"))
w8_label_conf = widgets.FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                                    description="Label conf:", readout_format=".2f",
                                    style={"description_width": "130px"},
                                    layout=widgets.Layout(width="380px"))
w8_imgsz = widgets.Dropdown(options=[644, 1008, 1288], value=644, description="imgsz:",
                            style={"description_width": "130px"},
                            layout=widgets.Layout(width="240px"))
w8_nprev = widgets.IntSlider(value=6, min=1, max=16, description="Preview N:",
                             style={"description_width": "130px"},
                             layout=widgets.Layout(width="360px"))
w8_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "130px"},
                          layout=widgets.Layout(width="200px"))
w8_status = widgets.HTML()
w8_out = widgets.Output()


def _cutouts_df():
    df = STAGE2.get("cutouts")
    if df is None:
        p = DIR_CUTOUTS / "cutouts.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE2["cutouts"] = df
    if df is None:
        return pd.DataFrame()
    return df[df["cutout_path"].notna()] if "cutout_path" in df.columns else df


def replay_whole_mask(row):
    """Stage-1b mask -> cutout coordinates, via the Stage-2 canvas geometry."""
    arrays, _ = read_mask_npz(row["whole_mask_path"])
    src = arrays["mask"]
    ch, cw = int(row["canvas_h"]), int(row["canvas_w"])
    whole = np.zeros((ch, cw), dtype=bool)
    ys, xs = np.nonzero(src)
    cy = ys - int(row["mask_bbox_y1"]) + int(row["canvas_offset_y"])
    cx = xs - int(row["mask_bbox_x1"]) + int(row["canvas_offset_x"])
    ok = (cy >= 0) & (cy < ch) & (cx >= 0) & (cx < cw)
    whole[cy[ok], cx[ok]] = True
    return whole


def render_preview8():
    with w8_out:
        clear_output(wait=True)
        cuts = _cutouts_df()
        if cuts.empty:
            print("!! No cutouts - run Cell 8 batch first.")
            return
        sample = cuts.sample(n=min(w8_nprev.value, len(cuts)), random_state=int(w8_seed.value))

        n = len(sample)
        ncol = 3
        nrow = int(np.ceil(n / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(_FW * 0.62 * ncol, _FW * 0.42 * nrow),
                                 squeeze=False)
        axes = axes.reshape(-1)
        recs, times = [], []
        for ax, (_, row) in zip(axes, sample.iterrows()):
            t0 = time.time()
            try:
                whole = replay_whole_mask(row)
                wa = int(whole.sum())
                regions, meta = {}, {}
                conf_map = {"cap": w8_cap_conf.value, "label": w8_label_conf.value}
                for key, prompt in (("cap", w8_cap_prompt.value),
                                    ("label", w8_label_prompt.value)):
                    res = sam3_preview(row["cutout_path"], [prompt], conf_map[key],
                                       w8_imgsz.value)
                    m = result_masks_bool(res)
                    confs, _ = result_meta(res)
                    union = m.any(axis=0) if m.shape[0] else np.zeros_like(whole)
                    if union.shape != whole.shape:
                        im = Image.fromarray(union.astype(np.uint8) * 255)
                        union = np.asarray(im.resize((whole.shape[1], whole.shape[0]),
                                                     Image.NEAREST)) > 127
                    union &= whole
                    regions[key] = union
                    meta[f"{key}_n_det"] = int(m.shape[0])
                    meta[f"{key}_conf"] = float(max(confs)) if confs else np.nan
                    meta[f"{key}_frac"] = float(union.sum()) / max(wa, 1)
                body = whole & ~regions["cap"] & ~regions["label"]
                meta["body_frac"] = float(body.sum()) / max(wa, 1)
                meta["class_name"] = row.get("class_name", "")
            except Exception as e:
                ax.text(0.5, 0.5, f"failed\n{e}", ha="center", fontsize=7)
                ax.axis("off")
                continue
            times.append(time.time() - t0)

            ax.imshow(np.array(Image.open(row["cutout_path"]).convert("RGB")))
            ax.contourf(body, levels=[0.5, 1], colors=[C_BODY], alpha=0.35)
            ax.contourf(regions["label"], levels=[0.5, 1], colors=[C_LABEL], alpha=0.35)
            ax.contourf(regions["cap"], levels=[0.5, 1], colors=[C_CAP], alpha=0.35)
            ax.set_title(f"{row.get('class_name', '')}\nlbl={meta['label_frac']:.2f}"
                         f"({meta['label_conf']:.2f}) cap={meta['cap_frac']:.2f}"
                         f"({meta['cap_conf']:.2f}) body={meta['body_frac']:.2f}", fontsize=7)
            ax.axis("off")
            recs.append(meta)
        for ax in axes[n:]:
            ax.axis("off")
        handles = [Line2D([0], [0], color=REGION_COLORS[c], lw=4, label=c.title())
                   for c in REGION_ORDER]
        _titled_legend(fig, f"Stage 3 Preview - cap='{w8_cap_prompt.value}' "
                            f"({w8_cap_conf.value:.2f})  label='{w8_label_prompt.value}' "
                            f"({w8_label_conf.value:.2f})", handles=handles, top_rect=0.88)
        plt.show()

        if recs:
            r = pd.DataFrame(recs)
            print(f"median label frac : {r['label_frac'].median():.3f}   "
                  f"median cap frac : {r['cap_frac'].median():.3f}   "
                  f"median body frac : {r['body_frac'].median():.3f}")
            if "class_name" in r.columns:
                display(HTML(r.groupby("class_name")
                             .agg(n=("body_frac", "size"),
                                  median_label=("label_frac", "median"),
                                  median_cap=("cap_frac", "median"))
                             .reset_index().to_html(index=False)))
            print("(no filtering here - all thresholds are set and applied in Cell 10)")
        spf = float(np.median(times)) if times else None
        STAGE3["sec_per_item"] = spf
        plan = resolve_compute_plan()
        eff = spf / plan["n_workers"] if (spf and plan["mode"] == "multi") else spf
        est = estimate_batch_runtime(eff, len(cuts))
        w8_status.value = status_html("stage3", n_scope=len(cuts), est=est)
        print(f"\nmedian {spf:.2f} s/cutout -> projected batch over {len(cuts):,}: {est}"
              if spf else "")


def lock8():
    params = {"cap_prompt": w8_cap_prompt.value.strip(),
              "label_prompt": w8_label_prompt.value.strip(),
              "cap_conf": float(w8_cap_conf.value), "label_conf": float(w8_label_conf.value),
              "imgsz": int(w8_imgsz.value)}
    lock_stage("stage3", params, n_preview=int(w8_nprev.value))
    w8_status.value = status_html("stage3", n_scope=len(_cutouts_df()))


def batch8():
    p = require_locked("stage3")
    cuts = _cutouts_df()
    if cuts.empty:
        print("!! No cutouts.")
        return
    need = ["cutout_id", "cutout_path", "whole_mask_path", "mask_bbox_x1", "mask_bbox_y1",
            "canvas_offset_x", "canvas_offset_y", "canvas_w", "canvas_h", "stem",
            "split", "class_name"]
    # item_id is required: run_sam3_batch matches it against the checkpoint key column
    # to skip already-processed cutouts on resume.
    items = [{**{k: r[k] for k in need}, "item_id": r["cutout_id"]} for _, r in cuts.iterrows()]

    with w8_out:
        clear_output(wait=True)
        print(f"Stage 3: {len(items):,} cutouts | cap='{p['cap_prompt']}' "
              f"({p['cap_conf']})  label='{p['label_prompt']}' ({p['label_conf']}) "
              f"imgsz={p['imgsz']}")
        df = run_sam3_batch("stage3", items, p, DIR_PSEUDOGT / "masks", key_col="item_id")
        if df.empty:
            print("!! No output produced.")
            return

        # Store all raw diagnostic columns unfiltered
        merged = cuts.merge(df, on="cutout_id", how="inner", suffixes=("", "_s3"))
        out_csv = DIR_PSEUDOGT / "pseudo_gt.csv"
        merged.to_csv(out_csv, index=False)
        STAGE3["pseudo_gt"] = merged

        n = len(merged)
        print(f"\nprocessed        : {n:,}")
        print(f"label detected   : {int((merged['label_frac'] > 0).sum()):,}")
        print(f"cap detected     : {int(merged['cap_present'].fillna(False).sum()):,}")
        errs = merged[merged.get("error", "").astype(str).str.len() > 0]
        print(f"errors           : {len(errs):,}")
        display(HTML(merged.groupby("class_name")
                     .agg(n=("cutout_id", "size"),
                          median_label=("label_frac", "median"),
                          median_cap=("cap_frac", "median"),
                          median_body=("body_frac", "median"))
                     .reset_index().to_html(index=False)))

        fig, axes = plt.subplots(1, 3, figsize=(_FW * 1.8, _FW * 0.42))
        for ax, col, c in zip(axes, ("label_frac", "cap_frac", "body_frac"),
                              (C_LABEL, C_CAP, C_BODY)):
            ax.hist(merged[col].dropna(), bins=40, color=c, alpha=0.85,
                    edgecolor="black", linewidth=0.3)
            ax.set_title(col.replace("_", " ").title())
            ax.set_xlabel("Area Fraction")
            ax.set_ylabel("Objects")
        _titled_legend(fig, "Stage 3 Region Area Fractions (raw, unfiltered - "
                            "thresholds are set in Cell 10)", top_rect=0.86)
        fig.savefig(DIR_FIGURES / "stage3_fractions.png", dpi=150, bbox_inches="tight")
        plt.show()

        log_params("stage3", outputs={"pseudo_gt": str(out_csv),
                                      "masks": str(DIR_PSEUDOGT / "masks")},
                   results={"n_processed": int(n),
                            "n_label_detected": int((merged["label_frac"] > 0).sum()),
                            "n_cap_detected": int(merged["cap_present"].fillna(False).sum()),
                            "n_errors": int(len(errs))})
        print(f"\npseudo_gt -> {out_csv}")


w8_bar, w8_buttons = make_action_bar(render_preview8, lock8, batch8,
                                     batch_label="Run Stage 3 batch")
for _w in (w8_cap_prompt, w8_label_prompt, w8_cap_conf, w8_label_conf, w8_imgsz):
    _w.observe(lambda ch: invalidate_lock("stage3", w8_buttons), names="value")

w8_status.value = status_html("stage3")
display(widgets.VBox([
    widgets.HBox([w8_cap_prompt, w8_label_prompt]),
    widgets.HBox([w8_cap_conf, w8_label_conf]),
    widgets.HBox([w8_imgsz, w8_nprev, w8_seed]),
    w8_bar,
    w8_status,
]), w8_out)

In [ ]:
# === Cell 10: Stage 4: Pseudo-GT Filter and Export ===
# Threshold tuning on the stored Stage 3 columns (no SAM3 re-run) and YOLO export.
# Nine thresholds: no body confidence gate (whole masks are annotated), no cap-presence gate.
# Export is one flat evaluation split; the source split is kept as a column.

STAGE4 = {"filtered": None, "export": None}

w9_lconf = widgets.FloatSlider(value=0.80, min=0.0, max=1.0, step=0.01,
                               description="label conf >=:", readout_format=".2f",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="410px"))
w9_cconf = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01,
                               description="cap conf >=:", readout_format=".2f",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="410px"))
w9_bfrac = widgets.FloatSlider(value=0.10, min=0.0, max=0.9, step=0.01,
                               description="min body frac:", readout_format=".2f",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="410px"))
w9_lfrac = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01,
                               description="min label frac:", readout_format=".2f",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="410px"))
w9_cfrac = widgets.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.01,
                               description="min cap frac:", readout_format=".2f",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="410px"))
w9_lndet_min = widgets.IntText(value=0, description="min label_n_det:",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="250px"))
w9_lndet_max = widgets.IntText(value=1, description="max label_n_det:",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="250px"))
w9_cndet_min = widgets.IntText(value=0, description="min cap_n_det:",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="250px"))
w9_cndet_max = widgets.IntText(value=99, description="max cap_n_det:",
                               style={"description_width": "150px"},
                               layout=widgets.Layout(width="250px"))
w9_gallery = widgets.IntSlider(value=12, min=0, max=48, description="Gallery N:",
                               style={"description_width": "140px"},
                               layout=widgets.Layout(width="380px"))
w9_eps = widgets.FloatSlider(value=0.002, min=0.0005, max=0.01, step=0.0005,
                             description="Polygon epsilon:", readout_format=".4f",
                             style={"description_width": "140px"},
                             layout=widgets.Layout(width="400px"))
w9_minpoly = widgets.IntText(value=24, description="Min poly px2:",
                             style={"description_width": "140px"},
                             layout=widgets.Layout(width="240px"))
w9_status = widgets.HTML()
w9_out = widgets.Output()


def _pseudo_gt_df():
    df = STAGE3.get("pseudo_gt")
    if df is None:
        p = DIR_PSEUDOGT / "pseudo_gt.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE3["pseudo_gt"] = df
    return df if df is not None else pd.DataFrame()


def _filtered():
    """The locked, filtered object set, loaded from disk if not in memory."""
    df = STAGE4.get("filtered")
    if df is None:
        p = DIR_TABLES / "filtered_objects.csv"
        if p.exists():
            df = pd.read_csv(p)
            STAGE4["filtered"] = df
    return df if df is not None else pd.DataFrame()


def apply_filter(df, p):
    """Apply all pseudo-GT thresholds (confidence, area fraction, detection count).

    Missing values fail a minimum (fillna(0)) but pass a maximum (fillna(99))."""
    m = (
        (df["label_conf"].fillna(0) >= p["label_conf_min"])
        & (df["cap_conf"].fillna(0) >= p["cap_conf_min"])
        & (df["body_frac"].fillna(0) >= p["min_body_area_frac"])
        & (df["label_frac"].fillna(0) >= p["min_label_area_frac"])
        & (df["cap_frac"].fillna(0) >= p["min_cap_area_frac"])
        & (df["label_n_det"].fillna(0) >= p["label_n_det_min"])
        & (df["label_n_det"].fillna(99) <= p["label_n_det_max"])
        & (df["cap_n_det"].fillna(0) >= p["cap_n_det_min"])
        & (df["cap_n_det"].fillna(99) <= p["cap_n_det_max"])
    )
    return df[m]


def _params9():
    return {"label_conf_min": float(w9_lconf.value),
            "cap_conf_min": float(w9_cconf.value),
            "min_body_area_frac": float(w9_bfrac.value),
            "min_label_area_frac": float(w9_lfrac.value),
            "min_cap_area_frac": float(w9_cfrac.value),
            "label_n_det_min": int(w9_lndet_min.value),
            "label_n_det_max": int(w9_lndet_max.value),
            "cap_n_det_min": int(w9_cndet_min.value),
            "cap_n_det_max": int(w9_cndet_max.value),
            "export_w": int(YOLO_INFER_W), "export_h": int(YOLO_INFER_H),
            "polygon_epsilon_frac": float(w9_eps.value),
            "min_polygon_area_px2": int(w9_minpoly.value)}


def mask_to_polygons(mask, min_area_px=None, epsilon_frac=None):
    """Boolean mask -> list of pixel-space polygons for YOLO segmentation labels.

    Contours are rejected by area, not vertex count (a rectangle has only 4 points).
    """
    min_area_px = int(w9_minpoly.value) if min_area_px is None else min_area_px
    epsilon_frac = float(w9_eps.value) if epsilon_frac is None else epsilon_frac
    m = (np.asarray(mask).astype(np.uint8)) * 255
    if not m.any():
        return []
    if cv2 is not None:
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        out = []
        for c in cnts:
            if len(c) < 3 or cv2.contourArea(c) < min_area_px:
                continue
            eps = epsilon_frac * cv2.arcLength(c, True)
            simplified = cv2.approxPolyDP(c, eps, True)
            if len(simplified) < 3:            # over-simplified: keep the original
                simplified = c
            out.append(simplified.reshape(-1, 2).astype(np.float64))
        return out
    # Fallback without OpenCV: per-component bounding polygon
    lbl, n = ndimage.label(mask)
    out = []
    for i in range(1, n + 1):
        ys, xs = np.nonzero(lbl == i)
        if xs.size < min_area_px:
            continue
        out.append(np.array([[xs.min(), ys.min()], [xs.max(), ys.min()],
                             [xs.max(), ys.max()], [xs.min(), ys.max()]], dtype=np.float64))
    return out


def resize_bool(mask, w, h):
    im = Image.fromarray(np.asarray(mask).astype(np.uint8) * 255)
    return np.asarray(im.resize((w, h), Image.NEAREST)) > 127


def write_yolo_label(path, regions, img_w, img_h):
    """Write YOLO polygon labels from already-resized region masks.

    A region with several disconnected parts writes one line per part."""
    lines = []
    for name in REGION_ORDER:
        mask = regions.get(name)
        if mask is None or not mask.any():
            continue
        cid = CLASS_MAP[name]
        for poly in mask_to_polygons(mask):
            norm = poly.copy()
            norm[:, 0] /= img_w
            norm[:, 1] /= img_h
            norm = np.clip(norm, 0.0, 1.0)
            coords = " ".join(f"{v:.6f}" for v in norm.reshape(-1))
            lines.append(f"{cid} {coords}")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text("\n".join(lines) + ("\n" if lines else ""))
    return len(lines)


def render_preview9():
    with w9_out:
        clear_output(wait=True)
        df = _pseudo_gt_df()
        if df.empty:
            print("!! No pseudo-GT - run Cell 9 batch first.")
            return
        p = _params9()
        kept = apply_filter(df, p)
        print(f"PASS: {len(kept):,} / {len(df):,}  ({100 * len(kept) / max(len(df), 1):.1f}%)")
        if len(kept):
            print(f"  cap present among kept : "
                  f"{100 * kept['cap_present'].fillna(False).mean():.1f}%"
                  f"   (recorded as a tag, not gated)")
            print(f"  median label frac      : {kept['label_frac'].median():.3f}")
            print(f"  median body frac       : {kept['body_frac'].median():.3f}")
            display(HTML(kept.groupby("class_name")
                         .agg(kept=("cutout_id", "size"),
                              median_body=("body_frac", "median"),
                              median_label=("label_frac", "median"),
                              median_cap=("cap_frac", "median"))
                         .reset_index().to_html(index=False)))
            display(HTML(kept.groupby("split").size().reset_index(name="kept")
                         .to_html(index=False)))
        n_nolabel = int((df["label_frac"].fillna(0) == 0).sum())
        print(f"  label_frac == 0 (no label found at all): {n_nolabel:,} "
              f"({100 * n_nolabel / max(len(df), 1):.1f}%)")

        # Sensitivity: sweep each threshold independently
        sweeps = [
            ("label_conf_min", np.arange(0.0, 1.0, 0.05)),
            ("cap_conf_min", np.arange(0.0, 1.0, 0.05)),
            ("min_body_area_frac", np.arange(0.0, 0.9, 0.05)),
            ("min_label_area_frac", np.arange(0.0, 0.5, 0.025)),
            ("min_cap_area_frac", np.arange(0.0, 0.5, 0.025)),
            ("label_n_det_max", np.arange(1, 9)),
            ("cap_n_det_max", np.arange(1, 9)),
        ]
        nrow_sw = 2
        ncol_sw = int(np.ceil(len(sweeps) / nrow_sw))
        fig, axes = plt.subplots(nrow_sw, ncol_sw, figsize=(_FW * 2.0, _FW * 0.75),
                                 squeeze=False)
        axes = axes.reshape(-1)
        for ax, (key, vals) in zip(axes, sweeps):
            ys = []
            for v in vals:
                q = dict(p)
                q[key] = v
                ys.append(len(apply_filter(df, q)))
            ax.plot(vals, ys, color="#2a9d8f", lw=1.5)
            ax.axvline(p[key], color="#d1495b", ls="--", lw=1)
            ax.set_title(key.replace("_", " "), fontsize=9)
            ax.set_xlabel("threshold")
            ax.set_ylabel("Objects Passing")
        for ax in axes[len(sweeps):]:
            ax.axis("off")
        _titled_legend(fig, "Filter Sensitivity - Each Threshold Swept Independently",
                       top_rect=0.90)
        fig.savefig(DIR_FIGURES / "filter_sensitivity.png", dpi=150, bbox_inches="tight")
        plt.show()

        if len(kept) and w9_gallery.value:
            n = min(int(w9_gallery.value), len(kept))
            sample = kept.sample(n=n, random_state=0)
            ncol = 4
            nrow = int(np.ceil(n / ncol))
            fig, axes = plt.subplots(nrow, ncol,
                                     figsize=(_FW * 0.48 * ncol, _FW * 0.34 * nrow),
                                     squeeze=False)
            axes = axes.reshape(-1)
            for ax, (_, row) in zip(axes, sample.iterrows()):
                # Image and mask loading are handled in separate try blocks
                try:
                    ax.imshow(np.array(Image.open(row["cutout_path"]).convert("RGB")))
                except Exception:
                    ax.text(0.5, 0.5, f"{row.get('cutout_id', '?')}\nimage failed",
                            ha="center", va="center", fontsize=6, color="#d1495b")
                    ax.axis("off")
                    continue

                mp = row.get("regions_mask_path")
                if pd.isna(mp) or not mp:
                    ax.set_title(f"{row.get('cutout_id', '?')}\nno regions mask recorded",
                                 fontsize=6, color="#d1495b")
                    ax.axis("off")
                    continue

                try:
                    arrays, _ = read_mask_npz(mp)
                    # Draw polygons body first so label/cap polygons stay on top
                    for key in REGION_ORDER:
                        m = arrays.get(key)
                        if m is None or not m.any():
                            continue
                        for poly in mask_to_polygons(m):
                            ax.add_patch(MplPolygon(poly, closed=True,
                                                    facecolor=REGION_COLORS[key], alpha=0.40,
                                                    edgecolor=REGION_COLORS[key],
                                                    linewidth=1.0))
                    ax.set_title(f"{row.get('class_name', '')}\n"
                                 f"body {row.get('body_frac', float('nan')):.2f}  "
                                 f"label {row.get('label_frac', float('nan')):.2f}"
                                 f"({row.get('label_conf', float('nan')):.2f})  "
                                 f"cap {row.get('cap_frac', float('nan')):.2f}", fontsize=6)
                except Exception as e:
                    ax.set_title(f"{row.get('cutout_id', '?')}\nmask read failed: "
                                 f"{type(e).__name__}", fontsize=6, color="#d1495b")
                ax.axis("off")
            for ax in axes[n:]:
                ax.axis("off")
            handles = [Line2D([0], [0], color=REGION_COLORS[c], lw=4, label=c.title())
                       for c in REGION_ORDER]
            _titled_legend(fig, "Filtered Objects - Polygon Masks", handles=handles,
                           top_rect=0.90)
            plt.show()

        STAGE4["filtered"] = kept
        w9_status.value = status_html("filter", n_scope=len(kept))


def lock9():
    p = _params9()
    df = _pseudo_gt_df()
    kept = apply_filter(df, p)
    STAGE4["filtered"] = kept
    out_csv = DIR_TABLES / "filtered_objects.csv"
    kept.to_csv(out_csv, index=False)
    lock_stage("filter", p, n_preview=int(w9_gallery.value))
    log_params("filter", outputs={"filtered_objects": str(out_csv)},
               results={"n_pool": int(len(df)), "n_kept": int(len(kept)),
                        "by_class": kept.groupby("class_name").size().to_dict()
                        if len(kept) else {}})
    print(f"filtered_objects -> {out_csv}  ({len(kept):,} objects)")
    w9_status.value = status_html("filter", n_scope=len(kept))


def batch9():
    """Export the pseudo-GT as a real YOLO segmentation dataset - single flat split."""
    p = require_locked("filter")
    kept = STAGE4.get("filtered")
    if kept is None or kept.empty:
        kept = _filtered()
    if kept is None or kept.empty:
        print("!! Nothing to export.")
        return
    with w9_out:
        clear_output(wait=True)
        img_dir = DIR_YOLO_DS / "images"
        lbl_dir = DIR_YOLO_DS / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        print(f"Exporting {len(kept):,} objects at {YOLO_INFER_W}x{YOLO_INFER_H} "
              f"(no train/val split - evaluation set)")

        rep = ProgressReporter("yolo_export", len(kept))
        rows, prov, n_lines = [], [], 0
        for _, row in kept.iterrows():
            rec = {"cutout_id": row["cutout_id"], "stem": row.get("stem", ""),
                   "source": "packwise", "class_name": row.get("class_name", ""),
                   "packwise_split": row.get("split", ""), "error": ""}
            try:
                arrays, _ = read_mask_npz(row["regions_mask_path"])
                src = Image.open(row["cutout_path"]).convert("RGB")
                img_r = src.resize((YOLO_INFER_W, YOLO_INFER_H), Image.BILINEAR)
                regions = {k: resize_bool(arrays[k], YOLO_INFER_W, YOLO_INFER_H)
                           for k in REGION_ORDER if k in arrays}
                img_dst = img_dir / f"{row['cutout_id']}.png"
                lbl_dst = lbl_dir / f"{row['cutout_id']}.txt"
                img_r.save(img_dst, format="PNG")
                nl = write_yolo_label(lbl_dst, regions, YOLO_INFER_W, YOLO_INFER_H)
                if nl == 0:
                    rec["error"] = "no polygons after simplification"
                n_lines += nl
                rec.update({"image_path": str(img_dst), "label_path": str(lbl_dst),
                            "n_lines": nl,
                            **{f"n_{k}_polys": len(mask_to_polygons(regions[k]))
                               for k in regions}})
                prov.append(provenance_record(
                    "yolo_export",
                    {"cutout": row["cutout_path"], "regions": row["regions_mask_path"]}, p,
                    {"cutout_id": row["cutout_id"], "source": "packwise",
                     "class_name": row.get("class_name", ""),
                     "packwise_split": row.get("split", ""),
                     "output_image": str(img_dst), "output_label": str(lbl_dst),
                     "pseudo_gt": "body/cap/label from SAM3 prompts"}))
            except Exception as e:
                rec["error"] = f"{type(e).__name__}: {e}"
                print(f"!! {row['cutout_id']}: {e}")
            rows.append(rec)
            rep.update()
        rep.close()

        exp = pd.DataFrame(rows)
        out_csv = DIR_YOLO_DS / "export_index.csv"
        exp.to_csv(out_csv, index=False)
        ok = exp[exp["error"] == ""]
        STAGE4["export"] = ok

        # Evaluation-only yaml: model.val(data=...) needs only the `val` key
        yaml_txt = (f"# PackWISE adapted to the KI-Sort body/cap/label convention\n"
                    f"# run_id: {RUN_ID}   config_hash: {CONFIG_HASH}\n"
                    f"# Evaluation set - single split, no train/val partition.\n"
                    f"path: {DIR_YOLO_DS}\nval: images\n\n"
                    f"nc: {len(CLASS_MAP)}\n"
                    f"names:\n" + "".join(f"  {v}: {k}\n"
                                          for k, v in sorted(CLASS_MAP.items(),
                                                             key=lambda kv: kv[1])))
        (DIR_YOLO_DS / "data.yaml").write_text(yaml_txt)

        card = [
            "# PackWISE - KI-Sort Adapted Segmentation Set", "",
            f"- run_id: `{RUN_ID}`", f"- config hash: `{CONFIG_HASH}`",
            f"- source dataset: `{DATASET_ROOT}`",
            "- source imagery: 586 RGB frames, 4096x4096, prism-based line-scan camera "
            "(SW-4000T-10GE) at 1278 lines/s over a 0.2 m/s conveyor, 12 halogen lamps",
            "- source licence: CC BY 4.0 (Fraunhofer, fordatis/463.2)",
            f"- objects exported: {len(ok):,}",
            f"- image size: {YOLO_INFER_W} x {YOLO_INFER_H} (w x h), native deployment "
            f"inference shape",
            "- single flat split; evaluation only, never fine-tune on this set",
            "- PackWISE's own train/val/test assignment is preserved per object in "
            "`export_index.csv` as `packwise_split`", "",
            "## Label provenance - READ BEFORE USING AS GROUND TRUTH", "",
            "| element | provenance |", "|---|---|",
            "| whole-object extent | dataset ground-truth COCO RLE instance mask |",
            "| body / cap / label split | **pseudo ground truth**: SAM3 concept prompts, "
            "body = whole - cap - label. No human annotation. |",
            "| polymer / material class | **absent**. PackWISE categories are material "
            "streams, not polymer types; this set supports segmentation generalizability "
            "only. |", "",
            "## In-scope criterion", "", IN_SCOPE_CRITERION, "",
            "## Per-category decisions", "",
            "| category | in scope | reason |", "|---|---|---|",
        ]
        for k, v in sorted(CATEGORY_DECISIONS.items(),
                           key=lambda kv: (not kv[1][0], kv[0])):
            card.append(f"| {k} | {'yes' if v[0] else 'no'} | {v[1]} |")
        card.append("")
        (DIR_YOLO_DS / "DATASET_CARD.md").write_text("\n".join(card))
        write_provenance_csv("yolo_export", prov)

        print(f"\nYOLO dataset -> {DIR_YOLO_DS}")
        print(f"  exported {len(ok):,}/{len(exp):,} objects | {n_lines:,} polygon rows")
        if len(ok):
            display(HTML(ok.groupby("class_name")
                         .agg(n=("cutout_id", "size"), mean_lines=("n_lines", "mean"))
                         .reset_index().to_html(index=False)))
        log_params("yolo_export",
                   outputs={"dataset": str(DIR_YOLO_DS),
                            "data_yaml": str(DIR_YOLO_DS / "data.yaml"),
                            "export_index": str(out_csv)},
                   results={"n_exported": int(len(ok)), "n_in": int(len(exp)),
                            "n_polygons": n_lines, "split": "single (evaluation only)"})
        w9_status.value = status_html("filter", n_scope=len(kept))


w9_bar, w9_buttons = make_action_bar(render_preview9, lock9, batch9,
                                     batch_label="Export YOLO dataset")
for _w in (w9_lconf, w9_cconf, w9_bfrac, w9_lfrac, w9_cfrac,
           w9_lndet_min, w9_lndet_max, w9_cndet_min, w9_cndet_max, w9_eps, w9_minpoly):
    _w.observe(lambda ch: invalidate_lock("filter", w9_buttons), names="value")

w9_status.value = status_html("filter")
display(widgets.VBox([
    widgets.HBox([w9_lconf, w9_cconf]),
    widgets.HBox([w9_bfrac, w9_lfrac, w9_cfrac]),
    widgets.HBox([w9_lndet_min, w9_lndet_max]),
    widgets.HBox([w9_cndet_min, w9_cndet_max]),
    widgets.HBox([w9_eps, w9_minpoly]),
    widgets.HBox([w9_gallery]),
    w9_bar,
    w9_status,
]), w9_out)

In [ ]:
# === Cell 11: YOLO Visual QA ===
# Shows deployment-model predictions beside the exported pseudo-GT for a sample of images.
# No metrics are computed (the deployment model is trained on all data); scoring is done in
# the benchmark notebook. Prediction polygons come from `result.masks.xy`.

QA = {"model": None}


def load_yolo():
    if QA["model"] is None:
        from ultralytics import YOLO
        QA["model"] = YOLO(str(YOLO_DEPLOY_WEIGHTS))
    return QA["model"]


def _export_df():
    df = STAGE4.get("export")
    if df is None:
        p = DIR_YOLO_DS / "export_index.csv"
        if p.exists():
            d = pd.read_csv(p)
            d["error"] = d["error"].fillna("")
            df = d[d["error"] == ""]
            STAGE4["export"] = df
    return df if df is not None else pd.DataFrame()


def parse_label_file(path, w, h):
    """YOLO polygon label file -> {region: [pixel-space polygons]}."""
    out = {k: [] for k in CLASS_MAP}
    for line in Path(path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        cid = int(float(parts[0]))
        name = CLASS_ID_TO_NAME.get(cid)
        if name is None:
            continue
        vals = np.array([float(v) for v in parts[1:]], dtype=np.float64)
        pts = vals.reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        out[name].append(pts)
    return out


w10_conf = widgets.FloatSlider(value=0.25, min=0.01, max=0.95, step=0.01,
                               description="YOLO conf:", readout_format=".2f",
                               style={"description_width": "120px"},
                               layout=widgets.Layout(width="360px"))
w10_iou = widgets.FloatSlider(value=0.50, min=0.1, max=0.9, step=0.05,
                              description="YOLO iou:", readout_format=".2f",
                              style={"description_width": "120px"},
                              layout=widgets.Layout(width="360px"))
w10_class = widgets.Dropdown(options=["all"], value="all", description="Class:",
                             style={"description_width": "120px"},
                             layout=widgets.Layout(width="280px"))
w10_nprev = widgets.IntSlider(value=4, min=1, max=10, description="Preview N:",
                              style={"description_width": "120px"},
                              layout=widgets.Layout(width="360px"))
w10_seed = widgets.IntText(value=0, description="Seed:", style={"description_width": "120px"},
                           layout=widgets.Layout(width="200px"))
w10_btn = widgets.Button(description="Run visual QA", button_style="info", icon="eye")
w10_refresh = widgets.Button(description="Refresh classes", icon="refresh")
w10_out = widgets.Output()


def _refresh_classes(_=None):
    exp = _export_df()
    opts = ["all"] + (sorted(exp["class_name"].dropna().unique()) if len(exp) else [])
    w10_class.options = opts
    if w10_class.value not in opts:
        w10_class.value = "all"


w10_refresh.on_click(_refresh_classes)


def _draw_polys(ax, polys_by_region, lw=1.3, alpha=0.25):
    for region, polys in polys_by_region.items():
        for p in polys:
            if len(p) < 3:
                continue
            closed = np.vstack([p, p[:1]])
            ax.fill(closed[:, 0], closed[:, 1], color=REGION_COLORS[region], alpha=alpha)
            ax.plot(closed[:, 0], closed[:, 1], color=REGION_COLORS[region], lw=lw)


def run_qa(_):
    with w10_out:
        clear_output(wait=True)
        exp = _export_df()
        if exp.empty:
            print("!! Nothing exported - run Cell 10 first.")
            return
        if w10_class.value != "all":
            exp = exp[exp["class_name"] == w10_class.value]
        if exp.empty:
            print("!! No exported objects for that class.")
            return
        model = load_yolo()
        sample = exp.sample(n=min(w10_nprev.value, len(exp)), random_state=int(w10_seed.value))
        n = len(sample)
        fig, axes = plt.subplots(n, 2, figsize=(_FW * 1.9, _FW * 0.42 * n), squeeze=False)
        counts = []
        for i, (_, r) in enumerate(sample.iterrows()):
            rgb = load_rgb(r["image_path"])
            h, w = rgb.shape[:2]

            gt = parse_label_file(r["label_path"], w, h)
            axes[i][0].imshow(rgb)
            _draw_polys(axes[i][0], gt)
            _panel_label(axes[i][0], "a", f"pseudo-GT  {r['class_name']}")
            axes[i][0].axis("off")

            res = model.predict(source=r["image_path"], conf=float(w10_conf.value),
                                iou=float(w10_iou.value), imgsz=YOLO_INFER_W,
                                rect=True, verbose=False)[0]
            pred = {k: [] for k in CLASS_MAP}
            if res.masks is not None and len(res.masks):
                cls = res.boxes.cls.cpu().numpy().astype(int)
                for poly, cid in zip(res.masks.xy, cls):
                    name = CLASS_ID_TO_NAME.get(int(cid))
                    if name is not None and len(poly) >= 3:
                        pred[name].append(np.asarray(poly, dtype=np.float64))
            axes[i][1].imshow(rgb)
            _draw_polys(axes[i][1], pred)
            _panel_label(axes[i][1], "b",
                         "prediction  " + "  ".join(f"{k}:{len(v)}" for k, v in pred.items()))
            axes[i][1].axis("off")
            counts.append({"item": str(r["cutout_id"])[:34], "class": r["class_name"],
                           **{f"gt_{k}": len(v) for k, v in gt.items()},
                           **{f"pred_{k}": len(v) for k, v in pred.items()}})
        handles = [Line2D([0], [0], color=REGION_COLORS[k], lw=6, label=k)
                   for k in ("body", "cap", "label")]
        _titled_legend(fig, f"Visual QA - Deployment Model on Exported Crops  "
                            f"conf={w10_conf.value:.2f}", handles=handles, ncol=3,
                       top_rect=0.90)
        plt.show()
        display(HTML(pd.DataFrame(counts).to_html(index=False)))
        print("\nRegion counts only. No IoU, no mAP, no aggregate score is computed here -")
        print("this model is the deployment tier and scoring belongs to the dedicated")
        print("cross-dataset benchmarking notebook.")


w10_btn.on_click(run_qa)
_refresh_classes()
display(widgets.VBox([
    widgets.HBox([w10_conf, w10_iou]),
    widgets.HBox([w10_class, w10_refresh]),
    widgets.HBox([w10_nprev, w10_seed]),
    w10_btn,
]), w10_out)

In [ ]:
# === Cell 12: Run Summary ===
# Funnel from annotations to exported objects, artifact list and locked parameters of every
# stage; written to summary.md and funnel.csv in the results directory.


def _safe_len(csv_path):
    try:
        return len(pd.read_csv(csv_path))
    except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError):
        return 0


def dir_size(path):
    n, tot = 0, 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            n += 1
            tot += p.stat().st_size
    return n, tot


def summarise_run(write_files=True):
    man = read_manifest()
    st = man.get("stages", {})
    expected = ["index", "stage1b", "stage2", "stage3", "filter", "yolo_export"]

    print("=" * 78)
    print(f"RUN SUMMARY  {RUN_ID}")
    print("=" * 78)
    print(f"config hash : {CONFIG_HASH}")
    print(f"dataset     : {DATASET_ROOT}")
    print(f"data out    : {DATA_OUT_ROOT}")
    print(f"results out : {RESULTS_OUT_ROOT}")
    print()

    def _res(stage, key, default=None):
        return st.get(stage, {}).get("results", {}).get(key, default)

    funnel = pd.DataFrame([
        {"step": "COCO annotations", "n": _res("index", "n_instances_total"),
         "note": "all instances across the selected splits"},
        {"step": "in scope", "n": _res("index", "n_instances_in_scope"),
         "note": "rigid plastic packaging only (per-category decisions)"},
        {"step": "whole-object masks", "n": _res("stage1b", "n_masks"),
         "note": "COCO RLE decoded, ground truth"},
        {"step": "cutouts", "n": _res("stage2", "n_cutouts"),
         "note": f"black canvas {CANVAS_ASPECT_W}:{CANVAS_ASPECT_H}"},
        {"step": "pseudo-GT computed", "n": _res("stage3", "n_processed"),
         "note": "SAM3 cap/label prompts, unfiltered"},
        {"step": "kept by filter", "n": _res("filter", "n_kept"),
         "note": "nine thresholds, Cell 10"},
        {"step": "exported objects", "n": _res("yolo_export", "n_exported"),
         "note": f"YOLO seg labels at {YOLO_INFER_W}x{YOLO_INFER_H}"},
    ])

    print("FUNNEL")
    print("-" * 78)
    prev = None
    for _, r in funnel.iterrows():
        n = r["n"]
        if n is None:
            print(f"  {r['step']:24s} {'--':>9}   {r['note']}")
            continue
        delta = ""
        if prev is not None and isinstance(n, (int, np.integer)) and prev:
            delta = f"  ({100 * n / prev:5.1f}% of prev)"
        print(f"  {r['step']:24s} {n:>9,}{delta}   {r['note']}")
        if isinstance(n, (int, np.integer)):
            prev = n
    print()

    print("ARTIFACTS")
    print("-" * 78)
    arts = []
    for label, path in [
        ("object index", DIR_TABLES / "object_index.csv"),
        ("object index (in scope)", DIR_TABLES / "object_index_in_scope.csv"),
        ("whole masks csv", DIR_STAGE1 / "whole_masks.csv"),
        ("whole masks", DIR_STAGE1 / "masks"),
        ("cutouts csv", DIR_CUTOUTS / "cutouts.csv"),
        ("cutout images", DIR_CUTOUTS / "cutouts"),
        ("pseudo-GT csv", DIR_PSEUDOGT / "pseudo_gt.csv"),
        ("pseudo-GT masks", DIR_PSEUDOGT / "masks"),
        ("filtered objects", DIR_TABLES / "filtered_objects.csv"),
        ("YOLO dataset", DIR_YOLO_DS),
        ("dataset card", DIR_YOLO_DS / "DATASET_CARD.md"),
        ("run manifest", MANIFEST_PATH),
    ]:
        p = Path(path)
        if not p.exists():
            continue
        if p.is_dir():
            n, tot = dir_size(p)
            arts.append({"artifact": label, "path": str(p), "items": n, "size": fmt_bytes(tot)})
        else:
            arts.append({"artifact": label, "path": str(p),
                         "items": _safe_len(p) if p.suffix == ".csv" else "",
                         "size": fmt_bytes(p.stat().st_size)})
    if arts:
        display(HTML(pd.DataFrame(arts).to_html(index=False)))

    if write_files:
        lines = [f"# Run Summary - {RUN_ID}", "",
                 f"- config hash: `{CONFIG_HASH}`",
                 f"- dataset: `{DATASET_ROOT}`",
                 f"- data out: `{DATA_OUT_ROOT}`",
                 f"- results out: `{RESULTS_OUT_ROOT}`",
                 f"- compute: {COMPUTE_PLAN['mode']} ({COMPUTE_PLAN['n_workers']} worker(s))",
                 ""]
        lines += ["## Funnel", "", "| step | n | note |", "|---|---:|---|"]
        for _, r in funnel.iterrows():
            n = r["n"]
            cell = f"{n:,}" if isinstance(n, (int, np.integer)) else "--"
            lines.append(f"| {r['step']} | {cell} | {r['note']} |")
        lines += ["", "## In-scope criterion", "", IN_SCOPE_CRITERION, ""]
        lines += ["### Per-category decisions", "",
                  "| category | in scope | reason |", "|---|---|---|"]
        for k, v in sorted(CATEGORY_DECISIONS.items(),
                           key=lambda kv: (not kv[1][0], kv[0])):
            lines.append(f"| {k} | {'yes' if v[0] else 'no'} | {v[1]} |")
        lines += ["", "## Label provenance", "",
                  "Whole-object extent is ground truth (COCO RLE instance mask). The "
                  "body/cap/label split is PSEUDO ground truth from SAM3 concept prompts. "
                  "PackWISE categories are material streams, not polymer types, so no "
                  "material labels are available.", ""]
        lines += ["## Locked parameters", ""]
        for s in expected:
            rec = st.get(s, {})
            if not rec.get("params"):
                continue
            lines.append(f"### {s}  (`{rec.get('param_hash')}`)")
            lines.append("")
            lines.append("```json")
            lines.append(json.dumps(rec["params"], indent=2, default=str))
            lines.append("```")
            lines.append("")
        lines += ["## Artifacts", "", "| artifact | path | items | size |", "|---|---|---:|---:|"]
        for a in arts:
            lines.append(f"| {a['artifact']} | `{a['path']}` | {a['items']} | {a['size']} |")
        sp = RESULTS_OUT_ROOT / "summary.md"
        sp.write_text("\n".join(lines))
        funnel.to_csv(DIR_TABLES / "funnel.csv", index=False)
        print(f"\nsummary -> {sp}")
        print(f"funnel  -> {DIR_TABLES / 'funnel.csv'}")

    never = [s for s in expected if s not in st]
    if never:
        print(f"\n!! never run: {', '.join(never)}")
    print("=" * 78)
    return funnel


w11_btn = widgets.Button(description="Summarise run", button_style="success",
                         icon="flag-checkered")
w11_out = widgets.Output()


def _run11(_):
    with w11_out:
        clear_output(wait=True)
        summarise_run()


w11_btn.on_click(_run11)
display(w11_btn, w11_out)